# Phase 6 · Encoder–decoder adapters and distilled CNN student

Model 2 trains residual adapters. Model 4 learns from teacher-generated training summaries. The optional gold-only control uses the same student architecture.

The completed runs’ cell outputs were recovered from the saved metrics, evaluator reports, and submission files.

## Required files and environment

- Attach `phase6_model2_student_assets.zip` or its extracted contents for the Phase 1 checkpoints and evaluator.
- Attach the official `Phase_6_Summ/` data, or enable Internet and set the `HF_TOKEN` secret to download it.
- Enable a GPU. To resume in a fresh session, restore the run folder and rerun setup before training.

The archive supplies the Phase 1 Model 2 and Model 4 assets, shared token cache, and teacher candidates.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.34,<1', 'ijson>=3.3,<4', 'numpy>=1.26,<3',
    'pandas>=2.2,<3', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5',
    'rouge-score==0.1.2', 'bert-score==0.3.13', 'transformers>=4.45,<5'], check=True)


In [ ]:
import base64
import contextlib
import copy
import gc
import hashlib
import heapq
import html
import importlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from urllib.parse import quote

import ijson
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display, HTML, FileLink
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PAD, UNK, BOS, EOS = range(4)
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for gpu in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu)
    print(f'GPU {gpu}: {props.name}, {props.total_memory / 2**30:.2f} GiB')


## 2. Settings

Model 2 uses its actual encoder-decoder path: source tokens enter the encoder; the decoder starts with BOS and learns the shifted reference summary, including EOS. This avoids the classification pooling/readout issue from earlier phases. The source budget stays at the Phase 1 encoder limit of 1,408 tokens; up to 383 summary text tokens plus EOS fit the 384-target limit. No causal language-model instruction is prepended to Model 2's encoder input.

The student retains eight original CNN blocks, including the large-dilation blocks, and trains all of its retained parameters. It uses the earlier causal summarization prompt, 1,536 selected source tokens and up to 384 summary text tokens. Both experiments use the same document groups, original tokenizer and evaluation protocol as the earlier notebooks.

Training has a 20-epoch maximum, patience 3 and a 10.5-hour session pause. Capacity testing selects runtime settings independently. No mandatory minimum epoch count is imposed.

Teacher selection defaults to the best compatible full held-out BERTScore, with ROUGE-L as a tie-breaker. Reported CodaBench scores are not used to select the teacher. Set `TEACHER_RUN_NAME='model1_full'` before teacher caching only if you deliberately want that teacher; the selected checkpoint is locked for the student run.


In [ ]:
VERSION = 'phase6_model2_student_v1'
DATA_VERSION = 'phase6_grouped_split_v2_blank_refs'
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phase6_model2_student')
ASSET_DIR = OUTPUT_ROOT / 'shared/assets'
ASSET_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_ZIP = 'phase6_model2_student_assets.zip'
MODEL_DIRS = {'model2': None, 'model4': None}
EXPECTED_CHECKPOINTS = {
    'model2': 'd4d73182b497b12ee9ff6eca43275d23844c24c5103d7209b4764e38b06a84de',
    'model4': 'fcab9f52a114bf8b281a219d57cb45e8ff892106c20257d2da45335fcd6d5dbb',
}
DATA_ROOT = None
SEARCH_ROOTS = []
REPO = 'Exploration-Lab/CS-781-Capstone'
DATA_REVISION = None
ID_FIELD, DOCUMENT_FIELD, SUMMARY_FIELD = 'id', 'document', 'summary'
SPLIT_SEED, VALIDATION_FRACTION, SELECTION_CASES = 42, 0.05, 64
PROMPT_PREFIX = 'Summarize the following legal document, preserving key facts, legal reasoning and the decision.\nDocument:\n'
PROMPT_SUFFIX = '\nSummary:\n'
TEACHER_RUN_NAME = None
RUN_SMALL_SAMPLE_CHECK = True
RUN_PER_DOCUMENT_BERTSCORE = True
RUN_GOLD_CONTROL = False
COMMON = dict(seed=42, epochs=20, patience=3, context_policy='head_tail',
    min_lr_ratio=0.1, warmup_fraction=0.06, weight_decay=0.01, clip=1.0,
    effective_batch=8, micro_batch=1, activation_checkpointing=True, loss_chunk=128,
    log_every=20, save_every=100, max_hours=10.5, amp_initial_scale=1024.0,
    max_consecutive_amp_skips=12, capacity_headroom_gib=1.5, initial_generation_baseline=True)
ADAPTER_CONFIG = dict(COMMON, source_tokens=1408, target_tokens=383,
    lr=1e-4, adapter_dim=64, adapter_dropout=0.05, student_keep=None)
STUDENT_CONFIG = dict(COMMON, source_tokens=1536, target_tokens=384, lr=2e-5,
    adapter_dropout=0.0, student_keep=[0, 2, 4, 6, 7, 8, 9, 15])
EXPERIMENTS = {
    'model2_adapters': dict(model='model2', method='adapters', config=copy.deepcopy(ADAPTER_CONFIG)),
    'model4_distilled': dict(model='model4', method='kd', config=copy.deepcopy(STUDENT_CONFIG)),
    'model4_gold_control': dict(model='model4', method='control', config=copy.deepcopy(STUDENT_CONFIG)),
}
DECODE_DEFAULT = dict(temperature=0.0, top_p=0.95, max_new_tokens=384, min_new_tokens=32,
    no_repeat_ngram=4, repetition_penalty=1.1, batch_size=4, seed=42)
print(json.dumps(dict(experiments=EXPERIMENTS, decoding=DECODE_DEFAULT), indent=2))


In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 2**20), b''):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':')).encode()).hexdigest()

def save_torch(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def unpack_inputs():
    roots = [INPUT_ROOT]
    for archive_path in sorted(INPUT_ROOT.rglob('*.zip')):
        if not archive_path.name.startswith(('phase6_', 'model3_lora_phase6_', 'model1_lorma_phase6_')):
            continue
        destination = OUTPUT_ROOT / 'imports' / archive_path.stem
        marker = destination / 'imported.json'
        identity = dict(bytes=archive_path.stat().st_size, path=str(archive_path),
                        modified=archive_path.stat().st_mtime_ns)
        if marker.is_file() and read_json(marker) == identity:
            roots.append(destination)
            continue
        with zipfile.ZipFile(archive_path) as archive:
            members = [item for item in archive.infolist() if not item.is_dir()
                       and Path(item.filename).parts
                       and (Path(item.filename).parts[0] == 'phase6_model_assets'
                            or Path(item.filename).parts[0].startswith('phase6_'))]
            if shutil.disk_usage(OUTPUT_ROOT).free < sum(item.file_size for item in members) + 256 * 2**20:
                raise OSError('Not enough working space to extract the attached Phase 6 ZIP.')
            for item in tqdm(members, desc=f'Import {archive_path.name}'):
                target = destination / item.filename
                if not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP member.')
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst, length=8 * 2**20)
        write_json(marker, identity)
        roots.append(destination)
    return roots

def restore_run():
    destination = RUN_DIR
    options = []
    for root in SEARCH_ROOTS:
        for path in root.rglob('signature.json'):
            if path.parent.resolve() == destination.resolve() or not (path.parent / 'latest.pt').is_file():
                continue
            try:
                if read_json(path).get('run_name') == RUN_NAME and read_json(path).get('version') == VERSION:
                    options.append(path.parent)
            except (ValueError, OSError):
                continue
    options = sorted(set(options))
    if not options or (destination / 'latest.pt').is_file():
        return
    if len({digest_file(folder / 'latest.pt') for folder in options}) != 1:
        raise ValueError('Attach one version of this run to resume.')
    files = [path for path in options[0].rglob('*') if path.is_file() and path.suffix != '.tmp']
    if shutil.disk_usage(OUTPUT_ROOT).free < sum(path.stat().st_size for path in files) + 256 * 2**20:
        raise OSError('Insufficient space to restore saved training state.')
    for path in files:
        target = destination / path.relative_to(options[0])
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and digest_file(target) != digest_file(path):
            raise ValueError(f'Existing file conflicts with attached backup: {target}')
        if not target.exists():
            shutil.copy2(path, target)
    print('Restored:', destination, flush=True)

def resolve_remaining_inputs():
    global SEARCH_ROOTS, MODEL_DIRS, DATA_ROOT
    SEARCH_ROOTS = unpack_inputs()
    options = sorted(set(path.parent for root in SEARCH_ROOTS for path in root.rglob('run_config.json')
        if all((path.parent / name).is_file() for name in ('best.pt', 'tokenizer.json'))))
    hashes = {}
    for name in ('model2', 'model4'):
        if MODEL_DIRS[name] is None:
            matches = []
            for folder in options:
                if folder not in hashes:
                    hashes[folder] = digest_file(folder / 'best.pt')
                if hashes[folder] == EXPECTED_CHECKPOINTS[name]:
                    matches.append(folder)
            if not matches:
                raise FileNotFoundError(f'Missing original {name} Phase 1 assets. Attach phase6_model2_student_assets.zip.')
            MODEL_DIRS[name] = matches[0]
        MODEL_DIRS[name] = Path(MODEL_DIRS[name])
        if digest_file(MODEL_DIRS[name] / 'best.pt') != EXPECTED_CHECKPOINTS[name]:
            raise ValueError(f'Wrong Phase 1 checkpoint for {name}.')
        print(name, MODEL_DIRS[name], flush=True)
    def valid(folder):
        return folder is not None and all((Path(folder) / name).is_file()
                                         for name in ('Train/train.json', 'Test/test.json'))
    if not valid(DATA_ROOT):
        options = sorted(set(path.parent.parent for root in SEARCH_ROOTS for path in root.rglob('train.json')
            if path.parent.name == 'Train' and 'Phase_6_Summ' in path.parts and valid(path.parent.parent)))
        if len({tuple(digest_file(folder / f) for f in ('Train/train.json', 'Test/test.json')) for folder in options}) > 1:
            raise ValueError('Different official data versions attached. Set DATA_ROOT explicitly.')
        DATA_ROOT = options[0] if options else None
    if not valid(DATA_ROOT):
        from huggingface_hub import HfApi, snapshot_download
        token = os.environ.get('HF_TOKEN')
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret('HF_TOKEN')
            except Exception:
                token = None
        try:
            revision = DATA_REVISION or HfApi(token=token).dataset_info(REPO).sha
            DATA_ROOT = Path(snapshot_download(repo_id=REPO, repo_type='dataset', revision=revision,
                allow_patterns=['Phase_6_Summ/**'], token=token)) / 'Phase_6_Summ'
            write_json(ASSET_DIR / 'download_provenance.json', dict(repo=REPO, revision=revision))
        finally:
            token = None
    DATA_ROOT = Path(DATA_ROOT)
    if not valid(DATA_ROOT):
        raise FileNotFoundError('Official Phase 6 train/test files are missing.')
    destination = ASSET_DIR / 'Phase_6_Summ'
    for name in ('Train/train.json', 'Test/test.json'):
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if DATA_ROOT.resolve() != destination.resolve() and (not target.is_file() or digest_file(target) != digest_file(DATA_ROOT / name)):
            shutil.copy2(DATA_ROOT / name, target)
    DATA_ROOT = destination
    print('Official data:', DATA_ROOT, flush=True)


def load_remaining_bases():
    global BASES, TOKENIZER
    BASES = {}
    report = {}
    for name in ('model2', 'model4'):
        folder = MODEL_DIRS[name]
        saved = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
        tok = ScratchByteBPE.from_file(folder / 'tokenizer.json')
        config = copy.deepcopy(saved['model_config'])
        tok_hash = digest_file(folder / 'tokenizer.json')
        if config['vocab_size'] != tok.get_vocab_size() or saved.get('tokenizer_sha256', tok_hash) != tok_hash:
            raise ValueError(f'Tokenizer mismatch: {name}')
        probe = MODEL_CLASSES[name](**config)
        probe.load_state_dict(saved['model'], strict=True)
        if not all(torch.isfinite(v).all().item() for v in saved['model'].values() if v.is_floating_point()):
            raise FloatingPointError(f'Non-finite Phase 1 weights: {name}')
        BASES[name] = dict(config=config, state=saved['model'], tokenizer=tok,
            tokenizer_sha256=tok_hash, checkpoint_sha256=digest_file(folder / 'best.pt'))
        report[name] = dict(config=config, checkpoint_sha256=BASES[name]['checkpoint_sha256'],
            parameters=sum(p.numel() for p in probe.parameters()), tokenizer=tok_hash)
        del probe, saved
        gc.collect()
    if len({b['tokenizer_sha256'] for b in BASES.values()}) != 1:
        raise ValueError('The shared token cache requires identical original tokenizers.')
    TOKENIZER = BASES['model2']['tokenizer']
    write_json(ASSET_DIR / 'phase1_models.json', report)
    print(json.dumps(report, indent=2), flush=True)


def use_experiment(name):
    global MODEL_NAME, MODEL_CLASS, BASE, CONFIG, DECODE, RUN_NAME, RUN_DIR, METHOD
    entry = EXPERIMENTS[name]
    MODEL_NAME, METHOD = entry['model'], entry['method']
    MODEL_CLASS, BASE = MODEL_CLASSES[MODEL_NAME], BASES[MODEL_NAME]
    CONFIG = copy.deepcopy(entry['config'])
    DECODE = copy.deepcopy(DECODE_DEFAULT)
    RUN_NAME = name
    RUN_DIR = OUTPUT_ROOT / 'runs' / name
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    return RUN_DIR


@contextlib.contextmanager
def teacher_context(info):
    names = ('MODEL_NAME', 'MODEL_CLASS', 'BASE', 'CONFIG', 'DECODE', 'RUN_NAME', 'RUN_DIR', 'METHOD')
    previous = {name: globals().get(name) for name in names}
    signature = info['signature']
    globals().update(MODEL_NAME=info['model'], MODEL_CLASS=MODEL_CLASSES[info['model']],
        BASE=dict(config=info['model_config'], tokenizer_sha256=info['tokenizer_sha256']),
        CONFIG=copy.deepcopy(signature['config']), DECODE=copy.deepcopy(signature['decode']),
        RUN_NAME=info['run_name'], RUN_DIR=Path(info['folder']), METHOD=info['method'])
    try:
        yield
    finally:
        globals().update(previous)


def select_teacher():
    candidates = []
    seen = set()
    for root in SEARCH_ROOTS:
        for path in root.rglob('teacher_info.json'):
            info = read_json(path)
            info['folder'] = str(path.parent.resolve())
            signature = info['signature']
            if signature['split'] != DATA_AUDIT['split_sha256'] or info['tokenizer_sha256'] != BASE['tokenizer_sha256']:
                continue
            checkpoint = path.parent / 'best.pt'
            if info['checkpoint_sha256'] in seen:
                continue
            if not checkpoint.is_file() or digest_file(checkpoint) != info['checkpoint_sha256']:
                raise ValueError(f'Teacher checkpoint is missing or altered: {path.parent}')
            score = info['metrics']['bertscore_f1']
            if not math.isfinite(score):
                raise ValueError('Invalid saved teacher validation score.')
            candidates.append(info)
            seen.add(info['checkpoint_sha256'])
    # The newly trained encoder-decoder is also eligible, after full held-out evaluation.
    folder = OUTPUT_ROOT / 'runs/model2_adapters'
    pointer = folder / 'validation_latest.json'
    if pointer.is_file() and (folder / 'best.pt').is_file():
        record = read_json(pointer)
        digest = digest_file(folder / 'best.pt')
        if record['checkpoint'] == digest:
            candidates.append(dict(folder=str(folder), run_name='model2_adapters', model='model2', method='adapters',
                model_config=BASES['model2']['config'], tokenizer_sha256=BASES['model2']['tokenizer_sha256'],
                checkpoint_sha256=digest, signature=read_json(folder / 'signature.json'), metrics=record['metrics']))
    if TEACHER_RUN_NAME is not None:
        candidates = [row for row in candidates if row['run_name'] == TEACHER_RUN_NAME]
    if not candidates:
        raise FileNotFoundError('No compatible evaluated teacher. Attach the generated teacher/student asset ZIP.')
    candidates.sort(key=lambda row: (-row['metrics']['bertscore_f1'], -row['metrics']['rougel_f1'], row['run_name']))
    selected = candidates[0]
    lock = ASSET_DIR / 'teacher_selection.json'
    if lock.is_file():
        old = read_json(lock)
        matching = [row for row in candidates if row['checkpoint_sha256'] == old['checkpoint_sha256']]
        if not matching:
            raise ValueError('The locked teacher is absent. Restore its assets; do not switch teachers within a student run.')
        selected = matching[0]
    write_json(lock, selected)
    table = pd.DataFrame([dict(run=row['run_name'], **row['metrics'], selected=row['checkpoint_sha256'] == selected['checkpoint_sha256']) for row in candidates])
    table.to_csv(ASSET_DIR / 'teacher_candidates.csv', index=False)
    display(table)
    print('Selected teacher:', selected['run_name'], flush=True)
    return selected


def cache_teacher_summaries():
    info = select_teacher()
    folder = OUTPUT_ROOT / 'teacher_cache' / info['checkpoint_sha256'][:12]
    with teacher_context(info):
        saved = torch.load(Path(info['folder']) / 'best.pt', map_location='cpu', weights_only=False)
        if saved['signature'] != info['signature']:
            raise ValueError('Packaged teacher metadata does not match its checkpoint.')
        device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        model = SummaryModel(saved['runtime'], initialize=False)
        model.load_state_dict(saved['model'], strict=True)
        del saved
        if METHOD in ('lora', 'lorma'):
            merge_low_rank(model.base.blocks)
        model = model.to(device).eval()
        count = sum(p.numel() for p in model.parameters())
        if count <= student_parameter_count():
            raise ValueError('Teacher must be larger than the reduced student.')
        try:
            prediction = generate_split(model, 'train', folder, info['checkpoint_sha256'])
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    lock = dict(folder=str(folder), teacher_checkpoint=info['checkpoint_sha256'],
        teacher_run=info['run_name'], teacher_parameters=count,
        prediction_sha256=digest_file(prediction), training_ids=object_hash([r['id'] for r in CASE_INDEX['train']]))
    write_json(OUTPUT_ROOT / 'teacher_cache/latest.json', lock)
    print('Teacher summaries ready:', prediction, flush=True)
    return lock


def student_parameter_count():
    entry = EXPERIMENTS['model4_distilled']
    probe = SummaryModel(entry['config'], model_name='model4', base_info=BASES['model4'], method='kd')
    count = sum(p.numel() for p in probe.parameters())
    del probe
    gc.collect()
    return count


def compare_remaining_runs():
    rows = []
    for name in EXPERIMENTS:
        folder = OUTPUT_ROOT / 'runs' / name
        pointer = folder / 'validation_latest.json'
        if not pointer.is_file():
            continue
        record = read_json(pointer)
        if record['checkpoint'] != digest_file(folder / 'best.pt'):
            raise ValueError(f'Stale validation: {name}')
        saved = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
        rows.append(dict(run=name, **record['metrics'], **saved['parameters'],
            best_epoch=saved['epoch'], best_step=saved['step'], checkpoint_bytes=(folder / 'best.pt').stat().st_size))
        del saved
    teacher_path = ASSET_DIR / 'teacher_selection.json'
    if teacher_path.is_file():
        teacher = read_json(teacher_path)
        rows.append(dict(run='teacher:' + teacher['run_name'], **teacher['metrics']))
    folder = OUTPUT_ROOT / 'comparison'
    folder.mkdir(exist_ok=True)
    table = pd.DataFrame(rows)
    table.to_csv(folder / 'validation_comparison.csv', index=False)
    if teacher_path.is_file() and (OUTPUT_ROOT / 'teacher_cache/latest.json').is_file():
        cache = teacher_cache_lock()
        student_row = next((row for row in rows if row['run'] == 'model4_distilled'), None)
        if student_row:
            write_json(folder / 'distillation.json', dict(teacher=teacher['run_name'],
                teacher_validation=teacher['metrics'], student_validation={k: student_row[k] for k in teacher['metrics']},
                teacher_parameters=cache['teacher_parameters'], student_parameters=student_row['total'],
                parameter_reduction=1 - student_row['total'] / cache['teacher_parameters'],
                bertscore_change=student_row['bertscore_f1'] - teacher['metrics']['bertscore_f1'],
                caveat='Architectures, capacity and adaptation objectives differ. Use the optional matched gold-only student control to isolate sequence KD.'))
    display(table)
    return table


In [ ]:
resolve_remaining_inputs()


## 3. Original tokenizer and architecture definitions


In [ ]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [ ]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [ ]:
def sliding_attention(q,k,v,window,chunk,dropout=0.0):
    b,h,t,d=q.shape; blocks=math.ceil(t/chunk); padded=blocks*chunk
    qpad=F.pad(q,(0,0,0,padded-t))
    query=qpad.reshape(b,h,blocks,chunk,d).permute(0,2,1,3,4).reshape(b*blocks,h,chunk,d)
    positions=torch.arange(blocks,device=q.device)[:,None]*chunk-window+1+torch.arange(window+chunk-1,device=q.device)[None,:]
    indices=positions.clamp(0,t-1)
    keys=k[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    values=v[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    qi=torch.arange(chunk,device=q.device)[:,None]
    kj=torch.arange(window+chunk-1,device=q.device)[None,:]
    relative=(kj>=qi)&(kj<=qi+window-1)
    valid=(positions>=0)&(positions<t)
    mask=(relative[None,:,:]&valid[:,None,:])[None,:,:,:].expand(b,-1,-1,-1).reshape(b*blocks,1,chunk,window+chunk-1)
    output=F.scaled_dot_product_attention(query,keys,values,attn_mask=mask,dropout_p=dropout)
    return output.reshape(b,blocks,h,chunk,d).permute(0,2,1,3,4).reshape(b,h,padded,d)[:,:,:t]

class MixedAttention(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.window=window
        self.chunk=query_chunk; self.global_layer=global_layer; self.dropout=dropout
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
    def forward(self,x,past=None,cache=False):
        b,t,w=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.dim).unbind(2)
        q,k,v=(a.transpose(1,2) for a in (q,k,v))
        if past is not None:
            if t!=1: raise ValueError('Incremental decoding expects one token')
            k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        p=self.dropout if self.training else 0.0
        if self.global_layer or past is not None:
            output=F.scaled_dot_product_attention(q,k,v,dropout_p=p,is_causal=past is None)
        else: output=sliding_attention(q,k,v,self.window,self.chunk,p)
        kv=None
        if cache:
            kv=(k,v) if self.global_layer else (k[:,:,-(self.window-1):].contiguous(),v[:,:,-(self.window-1):].contiguous())
        return self.proj(output.transpose(1,2).contiguous().view(b,t,w)),kv

class MixedBlock(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.attn=MixedAttention(width,heads,window,query_chunk,global_layer,dropout)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)
    def forward(self,x,past=None,cache=False):
        a,kv=self.attn(self.n1(x),past,cache); x=x+self.drop(a)
        return x+self.drop(self.ff(self.n2(x))),kv


class LocalGlobalLM(nn.Module):
    def __init__(self,vocab_size=32768,width=704,layers=16,heads=11,context=2048,window=256,
                 global_every=4,query_chunk=128,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(context,query_chunk,global_every,loss_chunk)<1 or window<2: raise ValueError('Invalid architecture limits')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,heads=heads,context=context,
            window=window,global_every=global_every,query_chunk=query_chunk,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([MixedBlock(width,heads,window,query_chunk,(i+1)%global_every==0,dropout) for i in range(layers)])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))
    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def hidden(self,ids,past=None,position_offset=0,cache=False):
        t=ids.shape[1]; w=self.config['width']
        if position_offset+t>self.config['context']: raise ValueError('Context exceeded')
        pos=torch.arange(position_offset,position_offset+t,device=ids.device).float()
        freq=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        phase=pos[:,None]*freq[None,:]
        pe=torch.stack((phase.sin(),phase.cos()),dim=-1).flatten(1)[:,:w]
        # Match sinusoidal amplitude with scaled learned token embeddings.
        x=self.drop(self.token(ids)*math.sqrt(w)+pe.to(self.token.weight.dtype))
        caches=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,kv=block(x,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.norm(x),caches
    def forward(self,ids,targets=None,past=None,position_offset=0,cache=False):
        h,caches=self.hidden(ids,past,position_offset,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),caches
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b): return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [ ]:
class EncDecAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.dropout=dropout
        self.q=nn.Linear(width,width); self.k=nn.Linear(width,width)
        self.v=nn.Linear(width,width); self.out=nn.Linear(width,width)

    def forward(self,x,memory=None,valid=None,causal=False,past=None,cache=False):
        b,t,w=x.shape
        def split(z): return z.view(b,-1,self.heads,self.dim).transpose(1,2)
        q=split(self.q(x))
        if memory is not None and past is not None:
            k,v=past
        else:
            source=x if memory is None else memory
            k,v=split(self.k(source)),split(self.v(source))
            if past is not None:
                k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        mask=None
        if valid is not None: mask=valid[:,None,None,:]
        if causal:
            offset=k.shape[2]-t
            allowed=torch.arange(k.shape[2],device=x.device)[None,:] <= (torch.arange(t,device=x.device)+offset)[:,None]
            mask=allowed[None,None,:,:] if mask is None else mask & allowed[None,None,:,:]
        h=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0)
        h=self.out(h.transpose(1,2).contiguous().view(b,t,w))
        return h, (k,v) if cache else None

class EncoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w)
        self.attn=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,valid):
        x=x+self.drop(self.attn(self.n1(x),valid=valid)[0])
        return x+self.drop(self.ff(self.n2(x)))

class DecoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w); self.n3=nn.LayerNorm(w)
        self.self_attn=EncDecAttention(w,h,p); self.cross=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,memory,source_valid,target_valid=None,past=None,cache=False):
        self_past,cross_past=(None,None) if past is None else past
        a,skv=self.self_attn(self.n1(x),valid=target_valid,causal=True,past=self_past,cache=cache)
        x=x+self.drop(a)
        a,ckv=self.cross(self.n2(x),memory=memory,valid=source_valid,past=cross_past,cache=cache)
        x=x+self.drop(a); x=x+self.drop(self.ff(self.n3(x)))
        return x, (skv,ckv) if cache else None


class EncoderDecoder(nn.Module):
    def __init__(self,vocab_size=32768,width=704,encoder_layers=6,decoder_layers=8,heads=11,dropout=0.0):
        super().__init__()
        self.config=dict(vocab_size=vocab_size,width=width,encoder_layers=encoder_layers,
                         decoder_layers=decoder_layers,heads=heads,dropout=dropout)
        self.token=nn.Embedding(vocab_size,width)
        self.encoder=nn.ModuleList([EncoderBlock(width,heads,dropout) for _ in range(encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(width,heads,dropout) for _ in range(decoder_layers)])
        self.enc_norm=nn.LayerNorm(width); self.dec_norm=nn.LayerNorm(width)
        self.drop=nn.Dropout(dropout); self.gradient_checkpointing=True
        self.apply(self.initialize)

    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self,ids,offset=0):
        w=self.config['width']; positions=torch.arange(offset,offset+ids.shape[1],device=ids.device).float()
        frequency=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        angle=positions[:,None]*frequency[None,:]
        pe=torch.stack((angle.sin(),angle.cos()),dim=-1).flatten(1)[:,:w]
        return self.drop(self.token(ids)+pe.to(self.token.weight.dtype))

    def encode(self,src):
        valid=src.ne(PAD); x=self.embed(src)
        for block in self.encoder:
            if self.training and self.gradient_checkpointing:
                x=checkpoint(block,x,valid,use_reentrant=False)
            else: x=block(x,valid)
        return self.enc_norm(x),valid

    def decode(self,ids,memory,source_valid,past=None,cache=False):
        offset=0 if past is None else past[0][0][0].shape[2]
        x=self.embed(ids,offset)
        valid=ids.ne(PAD) if past is None else None
        caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.gradient_checkpointing:
                x=checkpoint(lambda a,b,c,d,layer=block: layer(a,b,c,d)[0],
                    x,memory,source_valid,valid,use_reentrant=False)
            else:
                x,kv=block(x,memory,source_valid,valid,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.dec_norm(x),caches

    def forward(self,src,decoder_ids,labels=None):
        memory,valid=self.encode(src); h,_=self.decode(decoder_ids,memory,valid)
        if labels is None: return F.linear(h,self.token.weight)
        flat=h.reshape(-1,h.shape[-1]); labels=labels.reshape(-1)
        def loss_chunk(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32)
        for start in range(0,len(labels),128):
            a,b=flat[start:start+128],labels[start:start+128]
            loss=loss+(checkpoint(loss_chunk,a,b,use_reentrant=False) if self.training else loss_chunk(a,b))
        return loss


In [ ]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [ ]:
MODEL_CLASSES = dict(model1=DecoderLM, model2=EncoderDecoder, model3=LocalGlobalLM, model4=GatedConvLM)
load_remaining_bases()
use_experiment('model2_adapters')


## 4. Audit and shared split

Blank training references, including reported IDs `299` and `4799`, are excluded and logged. Identical normalized documents share a split. Training copies of test text are excluded. Test summary fields are never read. Compare the resulting split hash with the teacher metadata before distillation.


In [ ]:
def iter_records(path):
    with Path(path).open('rb') as handle:
        first = handle.read(1024).lstrip(b'\xef\xbb\xbf \n\r\t')[:1]
    with Path(path).open('rb') as handle:
        if first == b'[':
            yield from ijson.items(handle, 'item')
        elif first == b'{':
            for key, value in ijson.kvitems(handle, ''):
                if not isinstance(value, dict):
                    raise ValueError('Expected a list of records or a document-ID-to-record object.')
                yield dict(value, **({ID_FIELD: key} if ID_FIELD not in value else {}))
        else:
            for line in handle:
                if line.strip():
                    yield json.loads(line)


def text_field(record, key):
    value = record.get(key)
    if isinstance(value, list) and all(isinstance(part, str) for part in value):
        value = '\n'.join(value)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f'Document {record.get(ID_FIELD)!r}: expected nonempty {key!r}; '
                         f'type={type(value).__name__}, value={repr(value)[:160]}, keys={list(record)}')
    return value


def audit_data():
    signature = dict(version=DATA_VERSION, train=digest_file(DATA_ROOT / 'Train/train.json'),
        test=digest_file(DATA_ROOT / 'Test/test.json'), seed=SPLIT_SEED, fraction=VALIDATION_FRACTION,
        selection_cases=SELECTION_CASES, fields=[ID_FIELD, DOCUMENT_FIELD, SUMMARY_FIELD])
    report_path = ASSET_DIR / 'data_audit.json'
    if report_path.is_file():
        saved = read_json(report_path)
        if saved['signature'] != signature:
            raise ValueError('Writable data preparation differs. Use a new OUTPUT_ROOT or restore its original data.')
        for name, digest in saved['prepared_sha256'].items():
            if not (ASSET_DIR / name).is_file() or digest_file(ASSET_DIR / name) != digest:
                raise ValueError(f'Prepared corpus/index changed: {name}')
        return read_json(ASSET_DIR / 'split.json'), saved
    test_hashes, seen_ids, rows, stats = set(), {}, {}, {}
    excluded_empty_summaries = []
    for split in ('test', 'train'):
        source = DATA_ROOT / ('Test/test.json' if split == 'test' else 'Train/train.json')
        corpus = ASSET_DIR / f'corpus_{split}.jsonl'
        temporary = corpus.with_suffix('.partial')
        rows[split] = []
        counts = Counter()
        seen_ids[split] = {}
        with temporary.open('wb') as outgoing:
            for record in tqdm(iter_records(source), desc=f'Audit {split}'):
                counts['raw_records'] += 1
                identifier = record.get(ID_FIELD)
                if isinstance(identifier, bool) or not isinstance(identifier, (str, int)) or not str(identifier).strip():
                    raise ValueError('Document IDs must be nonempty strings or integers.')
                identifier = str(identifier)
                if split == 'train':
                    raw_summary = record.get(SUMMARY_FIELD)
                    if isinstance(raw_summary, list) and all(isinstance(part, str) for part in raw_summary):
                        raw_summary = '\n'.join(raw_summary)
                    if isinstance(raw_summary, str) and not raw_summary.strip():
                        counts['empty_summary_excluded'] += 1
                        excluded_empty_summaries.append(dict(id=identifier, reason='empty_reference_summary'))
                        continue
                document = text_field(record, DOCUMENT_FIELD)
                summary = text_field(record, SUMMARY_FIELD) if split == 'train' else None
                normalized = hashlib.sha256(' '.join(document.split()).encode()).hexdigest()
                identity = object_hash([document, summary])
                if identifier in seen_ids[split]:
                    if seen_ids[split][identifier] != identity:
                        raise ValueError(f'Conflicting repeated document ID in {split}: {identifier}')
                    counts['duplicate_ids'] += 1
                    continue
                seen_ids[split][identifier] = identity
                if split == 'train' and normalized in test_hashes:
                    counts['train_test_text_overlap_excluded'] += 1
                    continue
                if split == 'test':
                    test_hashes.add(normalized)
                case = dict(id=identifier, document=document)
                if summary is not None:
                    case['summary'] = summary
                encoded = (json.dumps(case, ensure_ascii=False) + '\n').encode()
                row = dict(id=identifier, normalized_hash=normalized, document_sha256=hashlib.sha256(document.encode()).hexdigest(),
                    summary_sha256=hashlib.sha256(summary.encode()).hexdigest() if summary is not None else None,
                    offset=outgoing.tell(), bytes=len(encoded), document_words=len(document.split()),
                    reference_words=len(summary.split()) if summary is not None else None)
                outgoing.write(encoded)
                rows[split].append(row)
                counts['retained_rows'] += 1
        temporary.replace(corpus)
        stats[split] = dict(counts)
    write_json(ASSET_DIR / 'excluded_empty_summaries.json', excluded_empty_summaries)
    groups = sorted({row['normalized_hash'] for row in rows['train']},
                    key=lambda value: hashlib.sha256(f'{SPLIT_SEED}:{value}'.encode()).hexdigest())
    if len(groups) < 2:
        raise ValueError('Need at least two distinct training documents for a held-out split.')
    number = min(len(groups) - 1, max(1, round(len(groups) * VALIDATION_FRACTION)))
    heldout = set(groups[:number])
    index = dict(train=[row for row in rows['train'] if row['normalized_hash'] not in heldout],
                 val=[row for row in rows['train'] if row['normalized_hash'] in heldout], test=rows['test'])
    # Multiple summaries of identical text remain in the same split.
    assert not ({r['normalized_hash'] for r in index['train']} & {r['normalized_hash'] for r in index['val']})
    selected = sorted(index['val'], key=lambda r: hashlib.sha256(f'{SPLIT_SEED}:selection:{r["id"]}'.encode()).hexdigest())[:SELECTION_CASES]
    index['selection'] = selected
    write_json(ASSET_DIR / 'split.json', index)
    truth = {row['id']: read_case(row, 'val')['summary'] for row in index['val']}
    write_json(ASSET_DIR / 'val_truth.json', truth)
    write_json(ASSET_DIR / 'selection_truth.json', {row['id']: truth[row['id']] for row in selected})
    names = ['split.json', 'corpus_train.jsonl', 'corpus_test.jsonl', 'val_truth.json', 'selection_truth.json',
             'excluded_empty_summaries.json']
    report = dict(signature=signature, raw_statistics=stats, counts={s: len(v) for s, v in index.items()},
        document_groups=len(groups), split_sha256=object_hash(index),
        empty_summary_ids=[row['id'] for row in excluded_empty_summaries],
        prepared_sha256={name: digest_file(ASSET_DIR / name) for name in names},
        policy='Blank training references are excluded and logged; whitespace-normalized document groups share a split; training copies of test documents are excluded; test summaries are never read.')
    write_json(report_path, report)
    return index, report


def read_case(row, split):
    path = ASSET_DIR / ('corpus_test.jsonl' if split == 'test' else 'corpus_train.jsonl')
    with path.open('rb') as handle:
        handle.seek(row['offset'])
        case = json.loads(handle.read(row['bytes']))
    if case['id'] != row['id']:
        raise ValueError('Prepared document offset/ID mismatch.')
    return case


def prepare_tokens():
    signature = dict(data=DATA_AUDIT['split_sha256'], tokenizer=BASE['tokenizer_sha256'], version=DATA_VERSION)
    folder = OUTPUT_ROOT / 'shared/token_cache'
    roots = [root for search in SEARCH_ROOTS for root in search.rglob('token_cache') if root.is_dir()] + [folder]
    for candidate in roots:
        manifest_path = candidate / 'manifest.json'
        if not manifest_path.is_file():
            continue
        manifest = read_json(manifest_path)
        if manifest.get('signature') != signature:
            continue
        for name, digest in manifest['sha256'].items():
            if not (candidate / name).is_file() or digest_file(candidate / name) != digest:
                raise ValueError('Attached token cache is corrupt.')
        print('Reusing token cache:', candidate, flush=True)
        return candidate
    folder.mkdir(parents=True, exist_ok=True)
    index = {}
    for split in ('train', 'val', 'test'):
        progress_path = folder / f'{split}_progress.json'
        state = read_json(progress_path) if progress_path.is_file() else dict(signature=signature, rows=[], documents=0, summaries=0)
        if state['signature'] != signature:
            raise ValueError('Partial token cache belongs to a different split/tokenizer.')
        document_path, summary_path = folder / f'{split}_documents.bin', folder / f'{split}_summaries.bin'
        for path, count in ((document_path, state['documents']), (summary_path, state['summaries'])):
            if count and (not path.is_file() or path.stat().st_size < 2 * count):
                raise ValueError('Partial token cache is missing data.')
        with document_path.open('r+b' if document_path.exists() else 'w+b') as docs, summary_path.open('r+b' if summary_path.exists() else 'w+b') as sums:
            docs.truncate(2 * state['documents']); docs.seek(0, 2)
            sums.truncate(2 * state['summaries']); sums.seek(0, 2)
            for position, row in enumerate(tqdm(CASE_INDEX[split], desc=f'Tokenize {split}')):
                if position < len(state['rows']):
                    if state['rows'][position]['id'] != row['id']:
                        raise ValueError('Partial token cache order changed.')
                    continue
                case = read_case(row, split)
                document = TOKENIZER.encode(case['document']).ids
                summary = TOKENIZER.encode(case['summary']).ids if split != 'test' else []
                if not document or (split != 'test' and not summary):
                    raise ValueError(f'Empty tokenization: {row["id"]}')
                if min(document + summary) < 0 or max(document + summary) >= min(65536, BASE['config']['vocab_size']):
                    raise ValueError('Tokenizer/model vocabulary mismatch.')
                np.asarray(document, dtype='<u2').tofile(docs)
                np.asarray(summary, dtype='<u2').tofile(sums)
                state['rows'].append(dict(id=row['id'], document_start=state['documents'], document_length=len(document),
                                         summary_start=state['summaries'], summary_length=len(summary)))
                state['documents'] += len(document); state['summaries'] += len(summary)
                if (position + 1) % 50 == 0:
                    docs.flush(); sums.flush(); os.fsync(docs.fileno()); os.fsync(sums.fileno())
                    write_json(progress_path, state)
            docs.flush(); sums.flush()
            write_json(progress_path, state)
        index[split] = state['rows']
    write_json(folder / 'index.json', index)
    names = ['index.json'] + [f'{s}_{kind}.bin' for s in ('train', 'val', 'test') for kind in ('documents', 'summaries')]
    write_json(folder / 'manifest.json', dict(signature=signature, sha256={name: digest_file(folder / name) for name in names}))
    return folder


def source_tokens(tokens, cfg):
    budget = cfg['source_tokens']
    if len(tokens) <= budget:
        return list(tokens)
    gap = TOKENIZER.encode('\n[...]\n').ids
    if cfg['context_policy'] == 'head':
        return list(tokens[:budget])
    chunks = 2 if cfg['context_policy'] == 'head_tail' else 4 if cfg['context_policy'] == 'spread' else 0
    if not chunks or budget <= (chunks - 1) * len(gap) + chunks:
        raise ValueError('Invalid context selection or insufficient source budget.')
    available = budget - (chunks - 1) * len(gap)
    widths = [available // chunks + (i < available % chunks) for i in range(chunks)]
    starts = np.linspace(0, len(tokens) - max(widths), chunks).round().astype(int)
    selected = []
    for i, (start, width) in enumerate(zip(starts, widths)):
        if i:
            selected += gap
        if i == chunks - 1:
            start = len(tokens) - width
        selected += list(tokens[start:start + width])
    return selected


def prompt_tokens(document, cfg):
    prefix = [BOS] + TOKENIZER.encode(PROMPT_PREFIX).ids
    suffix = TOKENIZER.encode(PROMPT_SUFFIX).ids
    return prefix + source_tokens(document, cfg) + suffix


class CausalSummaryDataset(Dataset):
    def __init__(self, split, cfg):
        self.split, self.cfg = split, cfg
        source_split = 'val' if split == 'selection' else split
        all_rows = TOKEN_INDEX[source_split]
        positions = {row['id']: i for i, row in enumerate(all_rows)}
        self.rows = [all_rows[positions[row['id']]] for row in CASE_INDEX[split]]
        self.documents = np.memmap(TOKEN_FOLDER / f'{source_split}_documents.bin', dtype='<u2', mode='r')
        summary_file = TOKEN_FOLDER / f'{source_split}_summaries.bin'
        self.summaries = np.memmap(summary_file, dtype='<u2', mode='r') if summary_file.stat().st_size else None
    def __len__(self):
        return len(self.rows)
    def document(self, i):
        row = self.rows[i]
        return self.documents[row['document_start']:row['document_start'] + row['document_length']].astype(np.int64).tolist()
    def prompt(self, i):
        return prompt_tokens(self.document(i), self.cfg)
    def target(self, i):
        row = self.rows[i]
        if self.summaries is None:
            return []
        start = row['summary_start']
        return self.summaries[start:start + min(row['summary_length'], self.cfg['target_tokens'])].astype(np.int64).tolist() + [EOS]
    def target_count(self, i):
        return min(self.rows[i]['summary_length'], self.cfg['target_tokens']) + 1
    def __getitem__(self, i):
        prompt, target = self.prompt(i), self.target(i)
        if not target:
            raise ValueError('The test dataset has no training targets.')
        combined = prompt + target
        x = torch.tensor(combined[:-1], dtype=torch.long)
        y = torch.tensor(combined[1:], dtype=torch.long)
        y[:len(prompt) - 1] = -100
        if len(x) > BASE['config']['context']:
            raise ValueError('Source and target exceed the original model context.')
        return x, y


def collate_summaries(samples):
    length = max(len(x) for x, _ in samples)
    ids = torch.full((len(samples), length), PAD, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    for i, (x, y) in enumerate(samples):
        ids[i, :len(x)], labels[i, :len(y)] = x, y
    return ids, labels


def token_report():
    report = {}
    for split in ('train', 'val', 'test'):
        rows = TOKEN_INDEX[split]
        lengths = np.asarray([r['document_length'] for r in rows])
        targets = np.asarray([r['summary_length'] for r in rows])
        gap_count = len(TOKENIZER.encode('\n[...]\n').ids) * (1 if CONFIG['context_policy'] == 'head_tail' else 3 if CONFIG['context_policy'] == 'spread' else 0)
        coverage = np.where(lengths <= CONFIG['source_tokens'], 1.0, (CONFIG['source_tokens'] - gap_count) / lengths)
        report[split] = dict(cases=len(rows), document_token_percentiles=np.percentile(lengths, [50, 90, 95, 99, 100]).tolist(),
            summary_token_percentiles=np.percentile(targets, [50, 90, 95, 99, 100]).tolist(),
            mean_document_coverage=float(coverage.mean()), truncated_summary_fraction=float((targets > CONFIG['target_tokens']).mean()) if split != 'test' else None)
    write_json(ASSET_DIR / 'token_length_and_coverage.json', report)
    print(json.dumps(report, indent=2), flush=True)
    return report


In [ ]:
CASE_INDEX, DATA_AUDIT = audit_data()
print(json.dumps(DATA_AUDIT, indent=2))


## 5. Reuse the shared token cache

The original tokenizers are identical by SHA-256. Complete source/reference tokens are cached once. Context selection and target truncation are applied by each experiment's dataset, not baked into the cache. The data report records their coverage and truncation differences.


In [ ]:
TOKEN_FOLDER = prepare_tokens()
TOKEN_INDEX = read_json(TOKEN_FOLDER / 'index.json')
length_report = token_report()


## 6. Adaptation and summary losses

Model 2 gets zero-initialized residual bottleneck adapters after the encoder attention/feed-forward sublayers and decoder self-attention/cross-attention/feed-forward sublayers. The original backbone stays frozen. Cross-attention remains active during training and generation. This is a Houlsby-style extension with an additional cross-attention adapter.

The student keeps eight pretrained CNN blocks and the original embedding/output weights. Its convolution caches preserve the retained blocks' original dilations. All retained parameters are fine-tuned. It is substantially smaller than the teacher; the notebook checks actual parameter counts before caching teacher outputs.

Loss is summary-token cross-entropy with EOS and ignored padding. Sequence KD replaces training reference targets with the teacher's generated summaries. Validation always uses the original reference summaries. The optional gold-only control uses the identical reduced initialization and recipe to compare this target choice.


In [ ]:
class LowRankLinear(nn.Module):
    def __init__(self, base, method, rank, alpha, dropout):
        super().__init__()
        self.base, self.method = base, method
        base.requires_grad_(False)
        self.scale = alpha / rank
        self.drop = nn.Dropout(dropout)
        columns = base.in_features if method == 'lora' else base.out_features
        self.A = nn.Parameter(base.weight.new_empty(rank, columns))
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        if self.method == 'lora':
            return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)
        if self.method == 'lorma':
            z = F.linear(x, self.base.weight)
            z = z + self.scale * F.linear(F.linear(self.drop(z), self.A), self.B)
            return z if self.base.bias is None else z + self.base.bias
        raise ValueError('Unknown adaptation method.')
    def merged(self):
        with torch.no_grad():
            delta = self.B.float() @ self.A.float()
            if self.method == 'lorma':
                delta = delta @ self.base.weight.float()
            self.base.weight.add_((self.scale * delta).to(self.base.weight.dtype))
        return self.base

def merge_low_rank(module):
    for name, child in list(module.named_children()):
        if isinstance(child, LowRankLinear):
            setattr(module, name, child.merged())
        else:
            merge_low_rank(child)

def autocast_context(device):
    return torch.autocast('cuda', dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()

def sync_cuda():
    for gpu in range(torch.cuda.device_count()):
        torch.cuda.synchronize(gpu)

def make_optimizer(model, cfg):
    groups = [dict(params=[p for p in model.parameters() if p.requires_grad and (p.ndim > 1) == decay],
                   weight_decay=cfg['weight_decay'] if decay else 0.0) for decay in (True, False)]
    return torch.optim.AdamW([group for group in groups if group['params']], lr=cfg['lr'])

def make_scheduler(optimizer, cfg):
    total = math.ceil(len(CASE_INDEX['train']) / cfg['effective_batch']) * cfg['epochs']
    warmup = max(1, round(total * cfg['warmup_fraction']))
    def factor(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, total - warmup))
        return cfg['min_lr_ratio'] + (1 - cfg['min_lr_ratio']) * (1 + math.cos(math.pi * progress)) / 2
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


In [ ]:
class ResidualAdapter(nn.Module):
    def __init__(self, width, bottleneck, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.down = nn.Linear(width, bottleneck)
        self.up = nn.Linear(bottleneck, width)
        self.dropout = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, hidden):
        return hidden + self.up(self.dropout(F.gelu(self.down(self.norm(hidden)))))


class EncoderAdapterBlock(nn.Module):
    def __init__(self, base, cfg, width):
        super().__init__()
        self.base = base
        self.after_attention = ResidualAdapter(width, cfg['adapter_dim'], cfg['adapter_dropout'])
        self.after_ff = ResidualAdapter(width, cfg['adapter_dim'], cfg['adapter_dropout'])

    def forward(self, x, valid):
        b = self.base
        x = self.after_attention(x + b.drop(b.attn(b.n1(x), valid=valid)[0]))
        return self.after_ff(x + b.drop(b.ff(b.n2(x))))


class DecoderAdapterBlock(nn.Module):
    def __init__(self, base, cfg, width):
        super().__init__()
        self.base = base
        self.after_self = ResidualAdapter(width, cfg['adapter_dim'], cfg['adapter_dropout'])
        self.after_cross = ResidualAdapter(width, cfg['adapter_dim'], cfg['adapter_dropout'])
        self.after_ff = ResidualAdapter(width, cfg['adapter_dim'], cfg['adapter_dropout'])

    def forward(self, x, memory, source_valid, target_valid=None, past=None, cache=False):
        b = self.base
        self_past, cross_past = (None, None) if past is None else past
        a, skv = b.self_attn(b.n1(x), valid=target_valid, causal=True, past=self_past, cache=cache)
        x = self.after_self(x + b.drop(a))
        a, ckv = b.cross(b.n2(x), memory=memory, valid=source_valid, past=cross_past, cache=cache)
        x = self.after_cross(x + b.drop(a))
        x = self.after_ff(x + b.drop(b.ff(b.n3(x))))
        return x, (skv, ckv) if cache else None


def install_teacher_low_rank(module, method, cfg):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LowRankLinear(child, method, cfg['rank'], cfg['alpha'], cfg['adapter_dropout']))
        else:
            install_teacher_low_rank(child, method, cfg)


class SummaryModel(nn.Module):
    def __init__(self, cfg, model_name=None, base_info=None, method=None, initialize=True):
        super().__init__()
        self.cfg = copy.deepcopy(cfg)
        self.model_name = model_name or MODEL_NAME
        self.method = method or METHOD
        info = base_info or BASE
        self.base = MODEL_CLASSES[self.model_name](**copy.deepcopy(info['config']))
        if initialize:
            self.base.load_state_dict(info['state'], strict=True)
        self.adapted = []
        if self.method == 'adapters':
            if self.model_name != 'model2':
                raise ValueError('Adapters in this notebook require Model 2.')
            self.base.requires_grad_(False)
            width = info['config']['width']
            for group, wrapper in (('encoder', EncoderAdapterBlock), ('decoder', DecoderAdapterBlock)):
                blocks = getattr(self.base, group)
                for index, block in enumerate(list(blocks)):
                    blocks[index] = wrapper(block, cfg, width)
                    self.adapted.append(f'{group}.{index}')
        elif self.method in ('lora', 'lorma'):
            self.base.requires_grad_(False)
            install_teacher_low_rank(self.base.blocks, self.method, cfg)
        elif self.method in ('kd', 'control'):
            if self.model_name != 'model4':
                raise ValueError('The reduced student must start from Model 4.')
            keep = cfg['student_keep']
            if sorted(set(keep)) != keep or min(keep) < 0 or max(keep) >= len(self.base.blocks):
                raise ValueError('Invalid student layer selection.')
            self.base.blocks = nn.ModuleList([self.base.blocks[i] for i in keep])
            self.base.dilations = [self.base.dilations[i] for i in keep]
            self.base.config['layers'] = len(keep)
            self.base.receptive_field = 1 + (self.base.config['kernel_size'] - 1) * sum(self.base.dilations)
            self.base.requires_grad_(True)
        elif self.method != 'full':
            raise ValueError(self.method)
        if self.model_name in ('model1', 'model2'):
            self.base.gradient_checkpointing = cfg['activation_checkpointing']
        else:
            self.base.config['activation_checkpointing'] = cfg['activation_checkpointing']

    def hidden(self, ids, past=None, position_offset=0, cache=False):
        if self.model_name == 'model3':
            return self.base.hidden(ids, past=past, position_offset=position_offset, cache=cache)
        if self.model_name == 'model4':
            if position_offset + ids.shape[1] > self.base.config['context']:
                raise ValueError('Student context exceeded.')
            return self.base.hidden(ids, past=past, cache=cache)
        if self.model_name == 'model2':
            raise ValueError('Model 2 has separate source and decoder streams.')
        if position_offset + ids.shape[1] > self.base.config['context']:
            raise ValueError('Context exceeded.')
        position = torch.arange(position_offset, position_offset + ids.shape[1], device=ids.device)
        x = self.base.drop(self.base.token(ids) + self.base.position(position))
        caches = []
        for i, block in enumerate(self.base.blocks):
            if self.training and self.base.gradient_checkpointing:
                x = checkpoint(lambda h, layer=block: layer(h)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    caches.append(kv)
        return self.base.final_norm(x), caches

    def forward(self, ids, labels=None, past=None, position_offset=0, cache=False):
        if self.model_name == 'model2':
            if labels is not None:
                length = self.cfg['source_tokens']
                source, decoder = ids[:, :length], ids[:, length:]
                memory, valid = self.base.encode(source)
                hidden, _ = self.base.decode(decoder, memory, valid)
                labels = labels[:, length:]
            else:
                if past is None:
                    memory, valid = self.base.encode(ids)
                    decoder = torch.full((len(ids), 1), BOS, dtype=torch.long, device=ids.device)
                    kv = None
                else:
                    memory, valid, kv = past
                    decoder = ids
                hidden, kv = self.base.decode(decoder, memory, valid, past=kv, cache=cache)
                return F.linear(hidden[:, -1], self.base.token.weight).float(), (memory, valid, kv)
        else:
            hidden, caches = self.hidden(ids, past, position_offset, cache)
            if labels is None:
                return F.linear(hidden[:, -1], self.base.token.weight).float(), caches
        valid = labels.reshape(-1) >= 0
        selected = hidden.reshape(-1, hidden.shape[-1])[valid]
        targets = labels.reshape(-1)[valid]
        total = hidden.new_zeros((), dtype=torch.float32)
        def loss_chunk(h, y):
            return F.cross_entropy(F.linear(h, self.base.token.weight).float(), y, reduction='sum')
        for start in range(0, len(targets), self.cfg['loss_chunk']):
            h, y = selected[start:start + self.cfg['loss_chunk']], targets[start:start + self.cfg['loss_chunk']]
            total = total + (checkpoint(loss_chunk, h, y, use_reentrant=False)
                             if self.training and h.requires_grad else loss_chunk(h, y))
        return dict(loss_sum=total[None], tokens=total.new_tensor([len(targets)]))


def make_model(cfg=None, device='cpu', saved=None, merged=False):
    model = SummaryModel(cfg or CONFIG)
    if saved is not None:
        model.load_state_dict(saved, strict=True)
    if merged and model.method in ('lora', 'lorma'):
        merge_low_rank(model.base.blocks)
    return model.to(device)


def parameter_report(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                adapted_modules=len(model.adapted),
                retained_blocks=list(model.cfg.get('student_keep') or []) if model.model_name == 'model4' else [],
                receptive_field=getattr(model.base, 'receptive_field', None))


def maximum_training_length(cfg):
    if MODEL_NAME == 'model2':
        if cfg['source_tokens'] > 1408 or cfg['target_tokens'] + 1 > 384:
            raise ValueError('Keep Model 2 within its Phase 1 encoder/target lengths.')
        return cfg['source_tokens'] + cfg['target_tokens'] + 1
    length = len(prompt_tokens([4] * cfg['source_tokens'], cfg)) + cfg['target_tokens']
    if length > BASE['config']['context']:
        raise ValueError('Source/target shape exceeds the Phase 1 context.')
    return length


def self_checks():
    cfg = dict(CONFIG, activation_checkpointing=True, adapter_dropout=0.0)
    seed_everything(cfg['seed'])
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(cfg, device)
    for text in ('Section 42: भारत ₹100.', 'a  b\n c', '<s> literal text'):
        assert TOKENIZER.decode(TOKENIZER.encode(text).ids) == text
    model.eval()
    ids = torch.tensor([[5, 6, 7, 8]], device=device)
    with torch.no_grad():
        if MODEL_NAME == 'model2':
            memory, valid = model.base.encode(ids)
            decoder = torch.tensor([[BOS, 9, 10]], device=device)
            full, _ = model.base.decode(decoder, memory, valid)
            _, kv = model.base.decode(decoder[:, :-1], memory, valid, cache=True)
            incremental, _ = model.base.decode(decoder[:, -1:], memory, valid, past=kv, cache=True)
            torch.testing.assert_close(full[:, -1], incremental[:, -1], rtol=3e-4, atol=3e-4)
            changed = decoder.clone(); changed[:, -1] = 11
            other = model.base.decode(changed, memory, valid)[0]
            torch.testing.assert_close(full[:, :-1], other[:, :-1], rtol=0, atol=1e-6)
            padded_memory, padded_valid = model.base.encode(F.pad(ids, (0, 3), value=PAD))
            padded = model.base.decode(decoder, padded_memory, padded_valid)[0]
            torch.testing.assert_close(full, padded, rtol=3e-4, atol=3e-4)
            # Zero-initialized adapters reproduce the original Phase 1 model exactly.
            plain = MODEL_CLASSES['model2'](**BASE['config']).to(device).eval()
            plain.load_state_dict(BASE['state'], strict=True)
            ref = plain.decode(decoder, *plain.encode(ids))[0]
            torch.testing.assert_close(full, ref, rtol=0, atol=1e-6)
            del plain
        else:
            full = model(ids)[0]
            _, kv = model(ids[:, :-1], cache=True)
            incremental = model(ids[:, -1:], past=kv, position_offset=ids.shape[1] - 1, cache=True)[0]
            torch.testing.assert_close(full, incremental, rtol=3e-4, atol=3e-4)
            changed = ids.clone(); changed[:, -1] = 12
            torch.testing.assert_close(model.hidden(ids)[0][:, :-1], model.hidden(changed)[0][:, :-1], rtol=0, atol=1e-6)
            assert len(model.base.blocks) < BASE['config']['layers']
    x, y = collate_summaries([SummaryDataset('train', cfg)[0]])
    model.train()
    output = model(x.to(device), y.to(device))
    (output['loss_sum'].sum() / output['tokens'].sum()).backward()
    assert int(output['tokens'].sum()) == int((y >= 0).sum())
    if MODEL_NAME == 'model2':
        assert all(p.grad is None for p in model.parameters() if not p.requires_grad)
        assert model.base.token.weight.grad is None
        assert any(p.grad is not None and p.grad.abs().sum() > 0 for n, p in model.named_parameters() if '.after_' in n and '.up.' in n)
        assert any(p.grad is not None and p.grad.abs().sum() > 0 for n, p in model.named_parameters() if '.after_cross.up.' in n)
    else:
        assert all(p.requires_grad for p in model.parameters())
        assert model.base.token.weight.grad is not None
    assert all(torch.isfinite(p.grad).all().item() for p in model.parameters() if p.grad is not None)
    report = dict(model=MODEL_NAME, method=METHOD, tokenizer_roundtrip=True, causality=True,
        incremental_cache=True, padding=True, summary_only_loss=True, gradients=True, parameters=parameter_report(model))
    write_json(RUN_DIR / 'self_checks.json', report)
    del model, output, x, y
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Passed:', report, flush=True)
    return report


In [ ]:
class SummaryDataset(CausalSummaryDataset):
    def __init__(self, split, cfg):
        super().__init__(split, cfg)
        self.model_name = MODEL_NAME
        self.method = METHOD
        self.pseudo = None
        if split == 'train' and self.method == 'kd':
            lock = teacher_cache_lock()
            rows = read_json(Path(lock['folder']) / 'prediction.json')
            validate_prediction_rows(rows, [row['id'] for row in CASE_INDEX['train']])
            self.pseudo = {row['id']: TOKENIZER.encode(row['summary']).ids for row in rows}
            if any(not tokens for tokens in self.pseudo.values()):
                raise ValueError('A teacher summary tokenized to an empty target.')

    def prompt(self, i):
        if self.model_name == 'model2':
            return source_tokens(self.document(i), self.cfg)
        return super().prompt(i)

    def target(self, i):
        if self.pseudo is not None:
            return self.pseudo[self.rows[i]['id']][:self.cfg['target_tokens']] + [EOS]
        return super().target(i)

    def target_count(self, i):
        return len(self.target(i))

    def __getitem__(self, i):
        if self.model_name != 'model2':
            return super().__getitem__(i)
        source, target = self.prompt(i), self.target(i)
        if not target:
            raise ValueError('Test cases have no training labels.')
        source = source + [PAD] * (self.cfg['source_tokens'] - len(source))
        ids = source + [BOS] + target[:-1]
        labels = [-100] * len(source) + target
        return torch.tensor(ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


def teacher_cache_lock():
    selection = read_json(ASSET_DIR / 'teacher_selection.json')
    pointer = read_json(OUTPUT_ROOT / 'teacher_cache/latest.json')
    if pointer['teacher_checkpoint'] != selection['checkpoint_sha256']:
        raise ValueError('Teacher cache belongs to a different selected teacher.')
    folder = Path(pointer['folder'])
    if digest_file(folder / 'prediction.json') != pointer['prediction_sha256']:
        raise ValueError('Teacher summaries changed after caching.')
    signature = read_json(folder / 'generation.json')
    if signature['cases'] != object_hash(CASE_INDEX['train']):
        raise ValueError('Teacher cache does not match this training split.')
    return pointer


def run_signature():
    signature = dict(version=VERSION, run_name=RUN_NAME, model=MODEL_NAME, method=METHOD,
        phase1_checkpoint=BASE['checkpoint_sha256'], tokenizer=BASE['tokenizer_sha256'],
        split=DATA_AUDIT['split_sha256'], token_cache=read_json(TOKEN_FOLDER / 'manifest.json'),
        config=CONFIG, decode=DECODE, prompt=[PROMPT_PREFIX, PROMPT_SUFFIX], torch=torch.__version__)
    if METHOD == 'kd':
        lock = teacher_cache_lock()
        signature['distillation'] = dict(kind='sequence_level_teacher_targets',
            checkpoint=lock['teacher_checkpoint'], predictions=lock['prediction_sha256'])
    return signature


## 7. Generation, official scoring, training and analysis


In [ ]:
def blocked_ngram_tokens(sequence, size):
    if size < 2 or len(sequence) < size - 1:
        return []
    prefix = tuple(sequence[-(size - 1):])
    return list({sequence[i + size - 1] for i in range(len(sequence) - size + 1)
                 if tuple(sequence[i:i + size - 1]) == prefix})


@torch.inference_mode()
def decode_batch(model, prompts, cfg, device, seed=42):
    if not prompts or len({len(p) for p in prompts}) != 1:
        raise ValueError('Generation batches must contain equal-length prompts.')
    length = len(prompts[0])
    if MODEL_NAME != 'model2' and length + cfg['max_new_tokens'] > BASE['config']['context']:
        raise ValueError('Prompt plus generation budget exceeds the Phase 1 context.')
    generator = torch.Generator(device=device).manual_seed(seed)
    ids = torch.tensor(prompts, dtype=torch.long, device=device)
    model.eval()
    output = [[] for _ in prompts]
    reasons = ['length_limit'] * len(prompts)
    finished = [False] * len(prompts)
    past = None
    for step in range(cfg['max_new_tokens']):
        with autocast_context(device):
            logits, past = model(ids, past=past, position_offset=0 if step == 0 else length + step - 1, cache=True)
        scores = logits.float()
        if not torch.isfinite(scores).all().item():
            raise FloatingPointError('Non-finite generation logits.')
        scores[:, [PAD, UNK, BOS]] = float('-inf')
        if step < cfg['min_new_tokens']:
            scores[:, EOS] = float('-inf')
        for row, sequence in enumerate(output):
            if sequence and cfg['repetition_penalty'] != 1:
                positions = torch.tensor(sorted(set(sequence)), device=device)
                values = scores[row, positions]
                scores[row, positions] = torch.where(values < 0, values * cfg['repetition_penalty'], values / cfg['repetition_penalty'])
            blocked = blocked_ngram_tokens(sequence, cfg['no_repeat_ngram'])
            if blocked:
                scores[row, blocked] = float('-inf')
        if not torch.isfinite(scores).any(-1).all().item():
            raise RuntimeError('Decoding constraints removed every candidate token.')
        if cfg['temperature'] == 0:
            next_ids = scores.argmax(-1)
        else:
            sorted_scores, order = (scores / cfg['temperature']).sort(-1, descending=True)
            probabilities = sorted_scores.softmax(-1)
            removed = probabilities.cumsum(-1) - probabilities >= cfg['top_p']
            probabilities = probabilities.masked_fill(removed, 0)
            sampled = torch.multinomial(probabilities, 1, generator=generator)
            next_ids = order.gather(-1, sampled).squeeze(-1)
        for row, token in enumerate(next_ids.tolist()):
            if finished[row]:
                continue
            if token == EOS:
                finished[row] = True
                reasons[row] = 'eos'
            else:
                output[row].append(token)
        if all(finished):
            break
        ids = next_ids[:, None]
    records = []
    for sequence, reason in zip(output, reasons):
        summary = TOKENIZER.decode(sequence).strip()
        if not summary:
            raise RuntimeError('Generated an empty summary. Completed batches are saved; inspect this prompt before resuming.')
        records.append(dict(summary=summary, generated_tokens=len(sequence), stop_reason=reason))
    return records


def validate_prediction_rows(rows, expected_ids):
    if not isinstance(rows, list) or len(rows) != len(expected_ids):
        raise ValueError('Wrong number of generated summaries.')
    ids = []
    for row in rows:
        if set(row) != {'id', 'summary'} or not isinstance(row['id'], str) or not isinstance(row['summary'], str) or not row['summary'].strip():
            raise ValueError('Each output needs an exact ID and a nonempty summary string.')
        ids.append(row['id'])
    if len(set(ids)) != len(ids) or set(ids) != set(expected_ids):
        raise ValueError('Missing, duplicate or unknown prediction IDs.')


def generate_split(model, split, out, checkpoint_hash, policy=None, decode=None):
    cfg = dict(CONFIG, context_policy=policy or CONFIG['context_policy'])
    decode = copy.deepcopy(decode or DECODE)
    signature = dict(version=VERSION, checkpoint=checkpoint_hash, model=MODEL_NAME, method=METHOD,
        tokenizer=BASE['tokenizer_sha256'], cases=object_hash(CASE_INDEX[split]),
        context={key: cfg[key] for key in ('context_policy', 'source_tokens', 'target_tokens')},
        prompt=[PROMPT_PREFIX, PROMPT_SUFFIX], decode=decode, inference='phase6_seq2seq_causal_v2')
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    metadata_path = out / 'generation.json'
    if metadata_path.is_file():
        if read_json(metadata_path) != signature:
            raise ValueError('Generation inputs changed. Use a different output folder.')
    elif any(out.iterdir()):
        raise ValueError('Generation folder is nonempty without metadata.')
    write_json(metadata_path, signature)
    details_path = out / 'details.json'
    completed = read_json(details_path) if details_path.is_file() else {}
    expected = [row['id'] for row in CASE_INDEX[split]]
    if set(completed) - set(expected):
        raise ValueError('Saved generation contains unexpected document IDs.')
    data = SummaryDataset(split, cfg)
    buckets = defaultdict(list)
    for i, row in enumerate(data.rows):
        if row['id'] not in completed:
            prompt = data.prompt(i)
            buckets[len(prompt)].append((i, prompt))
    device = next(model.parameters()).device
    started = time.perf_counter()
    for length in sorted(buckets, reverse=True):
        group = buckets[length]
        for start in range(0, len(group), decode['batch_size']):
            batch = group[start:start + decode['batch_size']]
            batch_started = time.perf_counter()
            result = decode_batch(model, [prompt for _, prompt in batch], decode, device,
                seed=decode['seed'] + int(object_hash([data.rows[i]['id'] for i, _ in batch])[:8], 16) % 1000000)
            batch_seconds = time.perf_counter() - batch_started
            for (i, prompt), generated in zip(batch, result):
                row = data.rows[i]
                completed[row['id']] = dict(**generated, prompt_tokens=len(prompt), document_tokens=row['document_length'],
                                           batch_seconds_per_document=batch_seconds / len(batch))
            write_json(details_path, completed)
            print(f'{split}: {len(completed)}/{len(data)} summaries', flush=True)
    rows = [dict(id=identifier, summary=completed[identifier]['summary']) for identifier in expected]
    validate_prediction_rows(rows, expected)
    write_json(out / 'prediction.json', rows)
    write_json(out / 'inference.json', dict(new_generation_seconds=time.perf_counter() - started,
        documents=len(rows), device=str(device), batch_size=decode['batch_size'],
        total_decode_seconds=sum(row.get('batch_seconds_per_document', 0) for row in completed.values()),
        checkpoint=checkpoint_hash, all_cached=not bool(buckets)))
    return out / 'prediction.json'


def run_evaluator(prediction_path, split, out):
    if split not in ('selection', 'val'):
        raise ValueError('Only held-out reference summaries may be evaluated.')
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    truth_path = ASSET_DIR / ('selection_truth.json' if split == 'selection' else 'val_truth.json')
    command = [sys.executable, str(EVALUATOR_PATH), '--truth', str(truth_path), '--prediction', str(prediction_path)]
    print(f'Professor evaluator: {split}', flush=True)
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, flush=True)
    if result.stderr:
        print(result.stderr[-4000:], flush=True)
    (out / 'prof_evaluation.txt').write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    write_json(out / 'evaluator_provenance.json', dict(script=digest_file(EVALUATOR_PATH),
        truth=digest_file(truth_path), prediction=digest_file(prediction_path), returncode=result.returncode))
    result.check_returncode()
    fields = {'bertscore_f1': r'BERTScore F1:\s*([\d.eE+-]+)', 'rouge1_f1': r'ROUGE-1:\s*([\d.eE+-]+)', 'rougel_f1': r'ROUGE-L:\s*([\d.eE+-]+)'}
    metrics = {}
    for key, pattern in fields.items():
        match = re.search(pattern, result.stdout)
        if not match or not math.isfinite(float(match.group(1))):
            raise RuntimeError('Could not read finite metrics from the official evaluator output.')
        metrics[key] = float(match.group(1))
    write_json(out / 'metrics.json', metrics)
    return metrics


def load_best_model():
    saved = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=False)
    if saved['signature'] != run_signature():
        raise ValueError('Best checkpoint/data/settings signature mismatch.')
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(saved['runtime'], device, saved['model'], merged=True).eval()
    return model, saved


def export_predictions(split):
    model, saved = load_best_model()
    digest = digest_file(RUN_DIR / 'best.pt')
    out = RUN_DIR / ('validation' if split == 'val' else 'test') / object_hash(dict(checkpoint=digest, decode=DECODE))[:12]
    try:
        prediction = generate_split(model, split, out, digest)
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if split == 'val':
        metrics = run_evaluator(prediction, split, out)
        write_json(RUN_DIR / 'validation_latest.json', dict(folder=str(out), checkpoint=digest, metrics=metrics))
        return out
    archive_path = out / f'{RUN_NAME}_submission.zip'
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction, 'prediction.json')
    with zipfile.ZipFile(archive_path) as archive:
        assert archive.namelist() == ['prediction.json'] and archive.testzip() is None
    print('CodaBench submission:', archive_path, flush=True)
    # Submissions are small; a data link avoids the server-root 404 issue.
    encoded = base64.b64encode(archive_path.read_bytes()).decode()
    display(HTML(f'<a download="{archive_path.name}" href="data:application/zip;base64,{encoded}">Download {archive_path.name}</a>'))
    return archive_path


In [ ]:
EVALUATOR_PATH = ASSET_DIR / 'evaluate_phase6.py'
EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for Phase 6.\n\nRequires: rouge-score, bert-score, torch.\nTruth JSON: {"doc_id":"reference summary", ...}\nPrediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]\n"""\nimport argparse,json,sys\ntry:\n    from rouge_score import tokenizers, rouge_scorer\n    import bert_score\nexcept ImportError as e:\n    print(\'Missing dependency. Install requirements.txt first.\', file=sys.stderr); raise\n\ndef load(path):\n    with open(path,encoding=\'utf-8\') as f: return json.load(f)\n\ndef lcs(a,b):\n    if len(a)>len(b): a,b=b,a\n    masks={}\n    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)\n    s=0\n    for t in b:\n        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)\n    return s.bit_count() if hasattr(int,\'bit_count\') else bin(s).count(\'1\')\n\ndef read_pred(x):\n    if isinstance(x,dict): out=x\n    elif isinstance(x,list):\n        out={}\n        for r in x:\n            if not isinstance(r,dict) or \'id\' not in r or \'summary\' not in r: raise ValueError(\'Each prediction item must contain id and summary.\')\n            if r[\'id\'] in out: raise ValueError(f\'Duplicate prediction ID: {r["id"]}\')\n            out[r[\'id\']]=r[\'summary\']\n    else: raise ValueError(\'Prediction must be a JSON object or list of objects.\')\n    for k,v in out.items():\n        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f\'Invalid summary for ID {k!r}.\')\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 6 summaries on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t=load(a.truth); p=read_pred(load(a.prediction))\n    if not isinstance(t,dict) or not t: raise ValueError(\'Truth must be a non-empty JSON object.\')\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]\n    scorer=rouge_scorer.RougeScorer([\'rouge1\'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)\n    r1=rl=0.0\n    for pr,rf in zip(preds,refs):\n        s=scorer.score(rf,pr); r1+=s[\'rouge1\'].fmeasure\n        pt=tok.tokenize(pr); rt=tok.tokenize(rf)\n        if not pt and not rt: x=1.0\n        elif not pt or not rt: x=0.0\n        else:\n            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)\n        rl+=x\n    n=len(t); r1/=n; rl/=n\n    try:\n        import torch; device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    except ImportError: device=None\n    _,_,f=bert_score.score(preds,refs,lang=\'en\',batch_size=8,verbose=False,**({\'device\':device} if device else {}))\n    print(f\'Documents: {n}\'); print(f\'BERTScore F1: {float(f.mean().item()):.6f}\'); print(f\'ROUGE-1: {r1:.6f}\'); print(f\'ROUGE-L: {rl:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding='utf-8')
print('Evaluator:', EVALUATOR_PATH)


In [ ]:
def hardware_signature():
    return dict(version=VERSION, checkpoint=BASE['checkpoint_sha256'], config=CONFIG,
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                torch=torch.__version__)


def capacity_check():
    if not torch.cuda.is_available():
        raise RuntimeError('Enable the Kaggle GPU for capacity testing and full training.')
    path = RUN_DIR / 'hardware.json'
    if path.is_file():
        saved = read_json(path)
        if saved['signature'] != hardware_signature():
            raise ValueError('Hardware/settings changed. Restore the original GPU setup or use a new run folder.')
        print('Saved runtime:', saved['selected'], flush=True)
        return saved['selected']
    maximum_length = maximum_training_length(CONFIG)
    results = []
    for checkpointing in (True, False):
        for micro in (1, 2, 4, 8):
            batch_size = micro * torch.cuda.device_count()
            if batch_size > CONFIG['effective_batch'] or CONFIG['effective_batch'] % batch_size:
                continue
            cfg = dict(CONFIG, micro_batch=micro, activation_checkpointing=checkpointing)
            print(f'Testing {micro} cases/GPU; checkpointing={checkpointing}', flush=True)
            model = net = optimizer = scaler = ids = labels = loss = output = norm = None
            try:
                gc.collect(); torch.cuda.empty_cache()
                seed_everything(cfg['seed'])
                model = make_model(cfg, 'cuda:0').train()
                net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
                optimizer = make_optimizer(model, cfg)
                scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
                ids = torch.randint(4, BASE['config']['vocab_size'], (batch_size, maximum_length), device='cuda:0')
                labels = torch.full_like(ids, -100)
                labels[:, -(cfg['target_tokens'] + 1):] = ids[:, -(cfg['target_tokens'] + 1):]
                accumulation = cfg['effective_batch'] // batch_size
                denominator = cfg['effective_batch'] * (cfg['target_tokens'] + 1)
                for gpu in range(torch.cuda.device_count()):
                    torch.cuda.reset_peak_memory_stats(gpu)
                durations, retries = [], 0
                while len(durations) < 4:
                    optimizer.zero_grad(set_to_none=True)
                    sync_cuda(); started = time.perf_counter()
                    for _ in range(accumulation):
                        with autocast_context('cuda'):
                            output = net(ids, labels)
                            loss = output['loss_sum'].sum() / denominator
                        if not torch.isfinite(loss).item():
                            raise FloatingPointError('Non-finite capacity loss.')
                        scaler.scale(loss).backward()
                        del output, loss
                        output = loss = None
                    scaler.unscale_(optimizer)
                    norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
                    if not torch.isfinite(norm).item():
                        retries += 1
                        scaler.update(new_scale=scaler.get_scale() / 2)
                        if retries >= cfg['max_consecutive_amp_skips']:
                            raise FloatingPointError('Repeated non-finite capacity gradients.')
                        continue
                    scaler.step(optimizer); scaler.update()
                    sync_cuda(); durations.append(time.perf_counter() - started)
                    retries = 0
                memory = []
                for gpu in range(torch.cuda.device_count()):
                    free, total = torch.cuda.mem_get_info(gpu)
                    peak = torch.cuda.max_memory_reserved(gpu)
                    memory.append(dict(gpu=gpu, reserved_gib=peak / 2**30,
                        headroom_gib=min(free, total - peak) / 2**30))
                passed = all(row['headroom_gib'] >= cfg['capacity_headroom_gib'] for row in memory)
                results.append(dict(micro_batch=micro, activation_checkpointing=checkpointing,
                    passed=passed, seconds_per_update=float(np.mean(durations[1:])), memory=memory))
            except torch.OutOfMemoryError:
                results.append(dict(micro_batch=micro, activation_checkpointing=checkpointing, passed=False, reason='CUDA OOM'))
            except FloatingPointError as error:
                results.append(dict(micro_batch=micro, activation_checkpointing=checkpointing, passed=False, reason=str(error)))
            finally:
                model = net = optimizer = scaler = ids = labels = loss = output = norm = None
                gc.collect(); torch.cuda.empty_cache()
            print(results[-1], flush=True)
    passing = [row for row in results if row['passed']]
    if not passing:
        write_json(path, dict(signature=hardware_signature(), results=results, selected=None))
        raise RuntimeError('No safe maximum-shape runtime passed. Inspect the results before training.')
    selected = min(passing, key=lambda row: row['seconds_per_update'])
    write_json(path, dict(signature=hardware_signature(), results=results, selected=selected))
    updates = math.ceil(len(CASE_INDEX['train']) / CONFIG['effective_batch'])
    print('Selected:', selected, flush=True)
    print(f'Estimated training-only epoch time: {selected["seconds_per_update"] * updates / 3600:.2f} hours. Generation/evaluation and checkpoints add time.', flush=True)
    return selected


def runtime_config(micro_batch=None, activation_checkpointing=None):
    saved = read_json(RUN_DIR / 'hardware.json')
    if saved['signature'] != hardware_signature() or not saved['selected']:
        raise ValueError('Run a passing capacity check on this hardware first.')
    cfg = dict(CONFIG, **{key: saved['selected'][key] for key in ('micro_batch', 'activation_checkpointing')})
    if micro_batch is not None:
        cfg['micro_batch'] = micro_batch
    if activation_checkpointing is not None:
        cfg['activation_checkpointing'] = activation_checkpointing
    if not isinstance(cfg['micro_batch'], int) or cfg['micro_batch'] < 1:
        raise ValueError('Invalid microbatch.')
    return cfg


@torch.no_grad()
def validation_nll(model, cfg, split='selection'):
    model.eval()
    dataset = SummaryDataset(split, cfg)
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 and next(model.parameters()).is_cuda else model
    total, tokens = 0.0, 0
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=cfg['micro_batch'] * max(1, torch.cuda.device_count()),
        shuffle=False, collate_fn=collate_summaries, num_workers=0)
    for ids, labels in loader:
        with autocast_context(device):
            output = net(ids.to(device), labels.to(device))
        total += float(output['loss_sum'].sum())
        tokens += int(output['tokens'].sum())
    loss = total / max(1, tokens)
    return dict(summary_nll=loss, perplexity=math.exp(min(loss, 30)), target_tokens=tokens)


def small_sample_check(updates=50, examples=8):
    cfg = runtime_config()
    cfg['adapter_dropout'] = 0.0
    seed_everything(cfg['seed'])
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(cfg, device)
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 and device.startswith('cuda') else model
    optimizer = make_optimizer(model, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=device.startswith('cuda'), init_scale=cfg['amp_initial_scale'])
    data = SummaryDataset('train', cfg)
    chosen = np.random.default_rng(cfg['seed']).choice(len(data), min(examples, len(data)), replace=False).tolist()
    denominator = sum(data.target_count(i) for i in chosen)
    batch_size = cfg['micro_batch'] * max(1, torch.cuda.device_count())
    history, completed, retries = [], 0, 0
    try:
        while completed < updates:
            model.train(); optimizer.zero_grad(set_to_none=True)
            total = 0.0
            for begin in range(0, len(chosen), batch_size):
                ids, labels = collate_summaries([data[i] for i in chosen[begin:begin + batch_size]])
                with autocast_context(device):
                    output = net(ids.to(device), labels.to(device))
                    loss = output['loss_sum'].sum() / denominator
                if not torch.isfinite(loss).item():
                    raise FloatingPointError('Non-finite tiny-sample loss.')
                scaler.scale(loss).backward()
                total += float(loss.detach())
                del ids, labels, output, loss
            scaler.unscale_(optimizer)
            norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
            if not torch.isfinite(norm).item():
                retries += 1
                if not device.startswith('cuda') or retries >= cfg['max_consecutive_amp_skips']:
                    raise FloatingPointError('Tiny-sample gradients failed.')
                scaler.update(new_scale=scaler.get_scale() / 2)
                continue
            scaler.step(optimizer); scaler.update()
            completed += 1; retries = 0
            if completed == 1 or completed % 10 == 0 or completed == updates:
                row = dict(update=completed, summary_nll=total, perplexity=math.exp(min(total, 30)))
                history.append(row); print(row, flush=True)
    finally:
        model = net = optimizer = scaler = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    report = dict(case_ids=[CASE_INDEX['train'][i]['id'] for i in chosen], history=history,
                  loss_decreased=history[-1]['summary_nll'] < history[0]['summary_nll'])
    write_json(RUN_DIR / 'small_sample_check.json', report)
    if not report['loss_decreased']:
        print('Tiny-sample loss did not decrease. Inspect this diagnostic before starting the long run.', flush=True)
    return report


def log_metric(row):
    with (RUN_DIR / 'metrics.jsonl').open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row) + '\n')
    print(row, flush=True)


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda']) != torch.cuda.device_count():
            raise ValueError('Resume with the same GPU count.')
        torch.cuda.set_rng_state_all(state['cuda'])


def check_resume_memory():
    if not torch.cuda.is_available():
        raise RuntimeError('Enable the GPU for this disposable resume check.')
    import traceback
    for name in ('last_exc', 'last_value'):
        error = getattr(sys, name, None)
        if isinstance(error, BaseException) and error.__traceback__:
            traceback.clear_frames(error.__traceback__)
    gc.collect(); torch.cuda.empty_cache()
    cfg = runtime_config(micro_batch=1, activation_checkpointing=True)
    path = RUN_DIR / 'latest.pt'
    saved = torch.load(path, map_location='cpu', weights_only=False)
    if saved['signature'] != run_signature():
        raise ValueError('Checkpoint/data signature mismatch.')
    model = net = optimizer = scaler = ids = labels = output = loss = None
    try:
        model = make_model(cfg, 'cuda:0', saved['model']).train()
        optimizer = make_optimizer(model, cfg)
        optimizer.load_state_dict(saved['optimizer'])
        scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
        scaler.load_state_dict(saved['scaler'])
        saved = None
        net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
        batch_size = min(torch.cuda.device_count(), cfg['effective_batch'])
        length = maximum_training_length(cfg)
        ids = torch.randint(4, BASE['config']['vocab_size'], (batch_size, length), device='cuda:0')
        labels = torch.full_like(ids, -100)
        labels[:, -(cfg['target_tokens'] + 1):] = ids[:, -(cfg['target_tokens'] + 1):]
        optimizer.zero_grad(set_to_none=True)
        for start in range(0, cfg['effective_batch'], batch_size):
            count = min(batch_size, cfg['effective_batch'] - start)
            with autocast_context('cuda'):
                output = net(ids[:count], labels[:count])
                loss = output['loss_sum'].sum() / (cfg['effective_batch'] * (cfg['target_tokens'] + 1))
            scaler.scale(loss).backward()
            del output, loss
            output = loss = None
        scaler.unscale_(optimizer)
        norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
        if not torch.isfinite(norm).item():
            raise FloatingPointError('Resume stress-test gradients were non-finite; saved checkpoints were not changed.')
        scaler.step(optimizer); scaler.update(); sync_cuda()
        print('Passed maximum-shape resume at microbatch 1/GPU with checkpointing. Saved checkpoints were not modified.', flush=True)
    finally:
        model = net = optimizer = scaler = saved = ids = labels = output = loss = None
        gc.collect(); torch.cuda.empty_cache()


def train_run(additional_steps=None, micro_batch=None, activation_checkpointing=None):
    cfg = runtime_config(micro_batch, activation_checkpointing)
    signature = run_signature()
    signature_path = RUN_DIR / 'signature.json'
    if signature_path.is_file() and read_json(signature_path) != signature:
        raise ValueError('Run settings/data changed. Restore the original recipe or use a new run folder.')
    write_json(signature_path, signature)
    write_json(RUN_DIR / 'config.json', dict(run_name=RUN_NAME, model=MODEL_NAME, method=METHOD, config=CONFIG, decode=DECODE))
    write_json(RUN_DIR / 'execution.json', cfg)
    seed_everything(cfg['seed'])
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(cfg, device)
    parameters = parameter_report(model)
    print('Parameters:', parameters, flush=True)
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 and device.startswith('cuda') else model
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=device.startswith('cuda'), init_scale=cfg['amp_initial_scale'])
    data = SummaryDataset('train', cfg)
    state = dict(epoch=0, position=0, step=0, best_bertscore=-1.0, bad_epochs=0, complete=False,
                 skipped=0, consecutive_amp_skips=0, initial_evaluated=False)
    latest = RUN_DIR / 'latest.pt'
    if latest.is_file():
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            raise ValueError('Resume signature mismatch.')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer']); scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler']); state = saved['progress']; restore_rng(saved['rng'])
        del saved
        print('Resumed:', state, flush=True)
    started = time.monotonic(); start_step = state['step']
    batch_size = cfg['micro_batch'] * max(1, torch.cuda.device_count())
    def cpu_state():
        return {key: value.detach().cpu() for key, value in model.state_dict().items()}
    def save_latest():
        save_torch(latest, dict(signature=signature, model=cpu_state(), optimizer=optimizer.state_dict(),
            scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), progress=copy.deepcopy(state),
            rng=rng_state(), runtime=cfg))
        write_json(RUN_DIR / 'progress.json', state)
    def evaluate(count_patience):
        optimizer.zero_grad(set_to_none=True)
        nll = validation_nll(model, cfg)
        eval_rng = rng_state()
        weights = cpu_state()
        temporary_model = None
        try:
            temporary_model = make_model(cfg, device, weights, merged=True).eval()
            out = RUN_DIR / 'selection' / f'step_{state["step"]:08d}'
            identity = object_hash(dict(signature=signature, step=state['step']))
            prediction = generate_split(temporary_model, 'selection', out, identity)
        finally:
            temporary_model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            restore_rng(eval_rng)
        metrics = run_evaluator(prediction, 'selection', out)
        improved = metrics['bertscore_f1'] > state['best_bertscore'] + 1e-8
        if improved:
            state['best_bertscore'] = metrics['bertscore_f1']; state['bad_epochs'] = 0
            save_torch(RUN_DIR / 'best.pt', dict(signature=signature, model=weights, runtime=cfg,
                step=state['step'], epoch=state['epoch'], metrics=metrics, parameters=parameters))
        elif count_patience:
            state['bad_epochs'] += 1
        log_metric(dict(event='validation', step=state['step'], epoch=state['epoch'],
            best_bertscore=state['best_bertscore'], bad_epochs=state['bad_epochs'], **nll, **metrics))
        model.train()
    try:
        if not state['initial_evaluated']:
            if CONFIG['initial_generation_baseline']:
                evaluate(False)
            state['initial_evaluated'] = True
            save_latest()
        while not state['complete'] and state['epoch'] < cfg['epochs']:
            order = np.random.default_rng(cfg['seed'] + state['epoch']).permutation(len(data))
            model.train()
            while state['position'] < len(order):
                chosen = order[state['position']:state['position'] + cfg['effective_batch']]
                denominator = sum(data.target_count(int(i)) for i in chosen)
                optimizer.zero_grad(set_to_none=True)
                update_loss = 0.0
                for begin in range(0, len(chosen), batch_size):
                    ids, labels = collate_summaries([data[int(i)] for i in chosen[begin:begin + batch_size]])
                    with autocast_context(device):
                        output = net(ids.to(device), labels.to(device))
                        loss = output['loss_sum'].sum() / denominator
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError('Non-finite training loss; inspect this batch.')
                    scaler.scale(loss).backward()
                    update_loss += float(loss.detach())
                    del ids, labels, output, loss
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
                if not torch.isfinite(norm).item():
                    state['skipped'] += 1; state['consecutive_amp_skips'] += 1
                    if not device.startswith('cuda'):
                        raise FloatingPointError('Non-finite CPU gradients.')
                    scaler.update(new_scale=scaler.get_scale() / 2)
                    print('AMP overflow; retrying the same cases at a lower loss scale.', flush=True)
                    if state['consecutive_amp_skips'] >= cfg['max_consecutive_amp_skips']:
                        save_latest()
                        raise FloatingPointError('Repeated AMP failures. Resume state saved; inspect before continuing.')
                    continue
                scaler.step(optimizer); scaler.update(); scheduler.step()
                state['consecutive_amp_skips'] = 0
                state['step'] += 1; state['position'] += len(chosen)
                if state['step'] == start_step + 1 or state['step'] % cfg['log_every'] == 0:
                    log_metric(dict(event='train', step=state['step'], epoch=state['epoch'], summary_nll=update_loss,
                        lr=optimizer.param_groups[0]['lr'], gradient_norm=float(norm), skipped=state['skipped'],
                        elapsed_minutes=(time.monotonic() - started) / 60))
                if state['step'] % cfg['save_every'] == 0:
                    save_latest()
                if (additional_steps is not None and state['step'] - start_step >= additional_steps) or time.monotonic() - started >= cfg['max_hours'] * 3600:
                    save_latest()
                    report = dict(**state, session_paused=True, session_seconds=time.monotonic() - started)
                    write_json(RUN_DIR / 'session_report.json', report)
                    print(report, flush=True)
                    return report
            state['epoch'] += 1; state['position'] = 0
            evaluate(True)
            state['complete'] = state['epoch'] >= cfg['epochs'] or state['bad_epochs'] >= cfg['patience']
            if state['complete']:
                state['stop_reason'] = 'epochs' if state['epoch'] >= cfg['epochs'] else 'early_stopping'
            save_latest()
        report = dict(**state, session_seconds=time.monotonic() - started, parameters=parameters)
        write_json(RUN_DIR / 'summary.json', report)
        print(report, flush=True)
        return report
    except KeyboardInterrupt:
        save_latest()
        print('Interrupted; latest.pt saved. Rerun the training cell to resume.', flush=True)
        raise
    finally:
        optimizer.zero_grad(set_to_none=True)
        model = net = optimizer = scheduler = scaler = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
def repeated_ngram_fraction(words, size):
    grams = [tuple(words[i:i + size]) for i in range(max(0, len(words) - size + 1))]
    return (len(grams) - len(set(grams))) / max(1, len(grams))


def analyze_validation():
    pointer = read_json(RUN_DIR / 'validation_latest.json')
    if pointer['checkpoint'] != digest_file(RUN_DIR / 'best.pt'):
        raise ValueError('Validation belongs to an older best checkpoint. Rerun validation first.')
    folder = Path(pointer['folder'])
    rows = read_json(folder / 'prediction.json')
    detail = read_json(folder / 'details.json')
    truth = read_json(ASSET_DIR / 'val_truth.json')
    validate_prediction_rows(rows, truth)
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    output = RUN_DIR / 'analysis' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    output.mkdir(parents=True, exist_ok=False)
    case_rows = []
    metadata = {row['id']: row for row in TOKEN_INDEX['val']}
    case_metadata = {row['id']: row for row in CASE_INDEX['val']}
    for record in rows:
        identifier = record['id']
        predicted = record['summary']; reference = truth[identifier]
        words = predicted.split(); ref_words = reference.split()
        score = scorer.score(reference, predicted)
        token_data = metadata[identifier]
        original = read_case(case_metadata[identifier], 'val')['document']
        document_words = original.split()
        document_ngrams = {tuple(document_words[i:i + 3]) for i in range(max(0, len(document_words) - 2))}
        summary_ngrams = [tuple(words[i:i + 3]) for i in range(max(0, len(words) - 2))]
        gap_count = len(TOKENIZER.encode('\n[...]\n').ids) * (1 if CONFIG['context_policy'] == 'head_tail' else 3 if CONFIG['context_policy'] == 'spread' else 0)
        coverage = 1.0 if token_data['document_length'] <= CONFIG['source_tokens'] else (CONFIG['source_tokens'] - gap_count) / token_data['document_length']
        case_rows.append(dict(id=identifier, generated_summary=predicted, reference_summary=reference,
            generated_words=len(words), reference_words=len(ref_words),
            generated_tokens=detail[identifier]['generated_tokens'], stop_reason=detail[identifier]['stop_reason'],
            document_tokens=token_data['document_length'], reference_tokens=token_data['summary_length'],
            reference_truncated_for_training=token_data['summary_length'] > CONFIG['target_tokens'],
            selected_context_fraction=coverage,
            rouge1_f1=score['rouge1'].fmeasure, rougel_f1=score['rougeL'].fmeasure,
            repeated_3gram_fraction=repeated_ngram_fraction(words, 3), repeated_4gram_fraction=repeated_ngram_fraction(words, 4),
            source_overlap_3gram_fraction=sum(gram in document_ngrams for gram in summary_ngrams) / max(1, len(summary_ngrams))))
    frame = pd.DataFrame(case_rows)
    if RUN_PER_DOCUMENT_BERTSCORE:
        import bert_score
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        _, _, scores = bert_score.score(frame['generated_summary'].tolist(), frame['reference_summary'].tolist(),
                                        lang='en', batch_size=8, device=device, verbose=False)
        frame['bertscore_f1'] = scores.cpu().numpy()
        del scores
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    frame.to_csv(output / 'per_document.csv', index=False)
    lengths = pd.cut(frame['document_tokens'], bins=[0, 1536, 4096, 8192, np.inf], labels=['up_to_1536', '1537_to_4096', '4097_to_8192', 'over_8192'])
    frame.assign(length_group=lengths).groupby('length_group', observed=True).agg(
        documents=('id', 'count'), rouge1_f1=('rouge1_f1', 'mean'), rougel_f1=('rougel_f1', 'mean'),
        context_coverage=('selected_context_fraction', 'mean'), repetition=('repeated_4gram_fraction', 'mean')).to_csv(output / 'document_length_groups.csv')
    manual = frame.sort_values('rougel_f1').head(min(20, len(frame))).copy()
    for column in ('facts_preserved', 'reasoning_preserved', 'decision_preserved', 'unsupported_claims', 'missed_information', 'notes'):
        manual[column] = ''
    manual.to_csv(output / 'manual_legal_review.csv', index=False)
    with (output / 'manual_review_documents.jsonl').open('w', encoding='utf-8') as handle:
        for identifier in manual['id']:
            case = read_case(case_metadata[identifier], 'val')
            handle.write(json.dumps(dict(case, generated_summary=next(r['summary'] for r in rows if r['id'] == identifier)), ensure_ascii=False) + '\n')
    history_path = RUN_DIR / 'metrics.jsonl'
    history = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()] if history_path.is_file() else []
    pd.DataFrame(history).to_csv(output / 'training_history.csv', index=False)
    validation = pd.DataFrame([row for row in history if row.get('event') == 'validation'])
    if not validation.empty:
        figure, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(validation['epoch'], validation['bertscore_f1'], marker='o'); axes[0].set(xlabel='Epoch', ylabel='Selection BERTScore F1')
        axes[1].plot(validation['epoch'], validation['summary_nll'], marker='o'); axes[1].set(xlabel='Epoch', ylabel='Selection summary NLL')
        figure.tight_layout(); figure.savefig(output / 'training_curves.png', dpi=150); plt.close(figure)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(frame['generated_words'], frame['rougel_f1'], s=8, alpha=0.5); axes[0].set(xlabel='Generated words', ylabel='ROUGE-L F1')
    axes[1].scatter(frame['selected_context_fraction'], frame['rouge1_f1'], s=8, alpha=0.5); axes[1].set(xlabel='Selected context fraction', ylabel='ROUGE-1 F1')
    figure.tight_layout(); figure.savefig(output / 'length_and_coverage.png', dpi=150); plt.close(figure)
    saved = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=False)
    write_json(output / 'summary.json', dict(full_validation=pointer['metrics'], selection_metrics=saved['metrics'],
        parameters=saved['parameters'], checkpoint_bytes=(RUN_DIR / 'best.pt').stat().st_size,
        best_epoch=saved['epoch'], best_step=saved['step'], progress=read_json(RUN_DIR / 'progress.json'),
        inference=read_json(folder / 'inference.json'), hardware=read_json(RUN_DIR / 'hardware.json'),
        source_policy=CONFIG['context_policy'], decode=DECODE,
        legal_review='Complete manual_legal_review.csv yourself. Lexical overlap and ROUGE are not factuality judgments.',
        limitations=['One fixed selection subset is reused for checkpoint selection.',
                    'Reference targets may be truncated for training; full references are used for evaluation.',
                    'Source coverage is constrained by the original context length.',
                    'Per-document ROUGE is diagnostic; aggregate metrics come from the professor evaluator.']))
    del saved
    print(frame[['generated_words', 'reference_words', 'rouge1_f1', 'rougel_f1', 'repeated_4gram_fraction']].describe().to_string(), flush=True)
    print('Analysis saved:', output, flush=True)
    return output


def run_generation_control(context_policy=None, max_new_tokens=None):
    policy = context_policy or CONFIG['context_policy']
    decode = dict(DECODE)
    if max_new_tokens is not None:
        decode['max_new_tokens'] = max_new_tokens
        decode['min_new_tokens'] = min(decode['min_new_tokens'], max_new_tokens - 1)
    model, saved = load_best_model()
    digest = digest_file(RUN_DIR / 'best.pt')
    identity = object_hash(dict(checkpoint=digest, policy=policy, decode=decode))[:12]
    out = RUN_DIR / 'generation_controls' / identity
    try:
        prediction = generate_split(model, 'selection', out, digest, policy=policy, decode=decode)
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    metrics = run_evaluator(prediction, 'selection', out)
    write_json(out / 'interpretation.json', dict(context_policy=policy, decode=decode,
        note='Inference-only control on the same checkpoint. This is not a separately trained technique or evidence of causation.'))
    return metrics


In [ ]:
for name in EXPERIMENTS:
    use_experiment(name)
    restore_run()
use_experiment('model2_adapters')


## 8. Model 2 — encoder-decoder adapters

Artifacts: `runs/model2_adapters`. Each cell selects this experiment explicitly. Capacity trials and the small-sample fit use disposable model copies; they do not alter saved training checkpoints.


In [ ]:
use_experiment('model2_adapters')
model2_adapters_checks = self_checks()


### Maximum-shape capacity and speed

Trials compare microbatches and activation checkpointing using forward, backward and successful optimizer updates. GPU headroom includes optimizer state. The fastest passing runtime is saved independently for this experiment. Validation/teacher decoding time is additional to the reported training-only estimate.


In [ ]:
use_experiment('model2_adapters')
model2_adapters_capacity = capacity_check()


### Small-sample fit


In [ ]:
use_experiment('model2_adapters')
if RUN_SMALL_SAMPLE_CHECK:
    model2_adapters_fit = small_sample_check()
    if not model2_adapters_fit['loss_decreased']:
        raise RuntimeError('Inspect the small-sample learning check before training.')


### Train or resume

Rerun this training cell after a session pause. `latest.pt` includes optimizer, scheduler, scaler, random state and position. `best.pt` is chosen by generated-summary BERTScore on the same fixed held-out selection subset. It may remain at step 0 if training never beats initialization. Three unimproved epochs stop the run.

An AMP overflow retries the same cases without advancing the data or learning-rate schedule. Repeated failures save the resume state and stop for inspection. For a resume OOM, select this experiment, run `check_resume_memory()`, then use `train_run(micro_batch=1, activation_checkpointing=True)` if it passes.


In [ ]:
use_experiment('model2_adapters')
model2_adapters_training = train_run()
print('Complete:', model2_adapters_training['complete'], flush=True)
if not model2_adapters_training['complete']:
    print('Paused. Rerun this training cell to continue; following cells use the current best.', flush=True)


Parameters: {'total': 125678976, 'trainable': 3322368, 'adapted_modules': 14, 'retained_blocks': [], 'receptive_field': None}
{'event': 'validation', 'step': 0, 'epoch': 0, 'best_bertscore': 0.786068, 'bad_epochs': 0, 'summary_nll': 11.055297140274575, 'perplexity': 63278.262217936506, 'target_tokens': 23855, 'bertscore_f1': 0.786068, 'rouge1_f1': 0.190231, 'rougel_f1': 0.121976}
{'event': 'train', 'step': 1, 'epoch': 0, 'summary_nll': 12.027403831481934, 'lr': 1.996007984031936e-07, 'gradient_norm': 10.401763916015625, 'skipped': 0, 'elapsed_minutes': 2.0075082057499998}
{'event': 'train', 'step': 20, 'epoch': 0, 'summary_nll': 10.555279731750488, 'lr': 2.095808383233533e-06, 'gradient_norm': 10.203741073608398, 'skipped': 0, 'elapsed_minutes': 2.1308890281833337}
{'event': 'train', 'step': 40, 'epoch': 0, 'summary_nll': 10.118141174316406, 'lr': 4.091816367265469e-06, 'gradient_norm': 10.330514907836914, 'skipped': 0, 'elapsed_minutes': 2.2625955062000003}
{'event': 'train', 'step': 

### Full held-out evaluation against original references


In [ ]:
use_experiment('model2_adapters')
model2_adapters_validation = export_predictions('val')


Professor evaluator: val
Documents: 350
BERTScore F1: 0.813701
ROUGE-1: 0.321547
ROUGE-L: 0.185564

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Analysis and review exports

Exports include per-document scores, source coverage, reference truncation, output length, repetition, training curves and resource usage. Manual review files contain source/reference/generated summaries and blank columns for legal facts, reasoning and the decision. Complete those judgments yourself; BERTScore and lexical overlap do not establish factuality.


In [ ]:
use_experiment('model2_adapters')
model2_adapters_analysis = analyze_validation()


### Test submission and download


In [ ]:
use_experiment('model2_adapters')
model2_adapters_submission = export_predictions('test')


CodaBench submission: /kaggle/working/phase6_model2_student/runs/model2_adapters/test/c740df302ccb/model2_adapters_submission.zip


<a download="model2_adapters_submission.zip" href="data:application/zip;base64,UEsDBBQAAAAIADhRYV18vXQ9djMAAIB6AgAPAAAAcHJlZGljdGlvbi5qc29u7X1bcxtHku77/gqEI06MHAFrcCe5+yTR8kgz9jmOkX1eFHpoAE2hPQCa090gzdjY/75166rMrKzqBkRdKNSDIkSy616V1y8z3/3HYPDf4t9g8F2x/u4/B9/NrhbT74b6N/Vht8uqB/nr3zb5ILu9zbfbbN8M7rN6kA2afC9/KG8Gjfir+MN6UOwH/z7kdVOU+4H8WX64L5tBvm+KZpuvB005yP/IV81O/GZw2K/zalCLH+X302fj79vOXqya5wM85k0h22d7+attscpUm5uyGuR3he4ATkT8KP//oSrFIOK/WTMQP26ytZjOoCo+bBo5lXK1Otw+oOmvi/r20OR69Cqvb0sxSTH8qtyL9a5z01nbZFtWzCrltG7Lus7rmkxM92t2brXNil2Op7fM871ekuynKneDTVGDzlAHZk8G9aFovL2Q7dR3qwfd6LVY9uC6PFSNXOWuEF2u1cTkJmfbwfJhUN/mq0L8d5tndzlotVKtsu22vEdthoPXr37+8T8HYzQtbz/gdSn2aL/vN8VqIxcvG23K7brYf0AfuJ17dzGdDV68H0zQYPpMxeRW/z4UlZld21geoLhe6Gpke/0Nam/uhNjCQpyzaIYnmpFLlO0f3C1Sm46GvcnuxIa1o+qByKT1DqnT/lDc5XvcJWz27nI8HbxUy36jp3Qr5qfuZFbn/lJMr/Ii61sj9mT5gLp8TZ4CXKoYfZf9S/db5eR9i37uswf5K/knfJnLw1YPvxF3h7nH6FDzale3Pd9W5V0h77b6TUsOZs8mlhz8LJu9kAdcF+qPij68uxqPBz++H0zRPAqf4sgulvk+vyncanKxd3j+2V7v3WCd5zvd0K1EzNl/0OIcxCvKa0j2xLQW44k4LvIe6HyqfFvkN+iKrB5wR6CH3aFWU4O93IqDWJW723xfa2JoyFKVNbnsd/F/BreCvIplHXbuBNbmr/L/gnbl9pau81WViy15Nx8vXrz/TnCA/xlS5jC/upqdxBxuimpnGQIlBfKPHr1FH2JGMcGMYog4DeycvP72uChn0UNZal7sydevczwbt46tfF05beFdD9fgNquafV55LeQXGzzMyfzPTcPngQ3eZMoPxQ2D+9f3xuOJMpzSP1f5m0z07m+F48F23zreMmJyYB4bwa/y/QfDEhSPFPRL7KNkMdngvhK/uM0bTVGWudhLTfUcn+zHNx3X8pfrqKJ8vZAcirWLH3PKGyXX1uzDY7+bfMvv6brQQ5jz0n9uz1QJGZTmqV7/flh/kP1IecytyN4ts7tunOeDH/XyWxbNygBgXn+pzWVQ/XEzL/DilTyWecIclSrFHyp1XEPM/WrETzluKn+Sa3xQL4djcUbMeGnZivdcDH/NlttcdldvynvBPA6CFWcfsmJfNx7TR13UpIt2Im6Civ0qcSA3nN49dvVUsocdmLXjxPDeWW5WZwXhZXDHgeyo+1qVd3n1APuW//+nIIPj4Wg0+mv7XmUzLZtcq63qIZpoRrq7FfdwTYRCTeDfMqz/n7I3ReVDgk573PUmq3IgUwjm1zxwrJnnbRfj6ejceFuEU/GsLcYMxX7X5f6LsjZOpEcCulvH/9sDtcdetGtBrqpSsI1qSFgBIf36+hPugAl0QPgDTNXbJ9o2zD3wYgJ0uZHMRE0trKfZg7zflOQXhWivJ7nLs73sHIjnU/BGX5a7pRBGfzNCrLq346vZhaeneeQnImFkNXlKYdnDfMexwVawJkqfe1HwaeDZyM3zVPaA4kaVxpb8UM0kzuzsCrVS7CbpnhaeYjZYHgolfARUmTDxN3plSNJfjMeXiRomapio4VOlhlL4VqK/XIYV8vHx7tQ+lUIjZQmpohrILgUMpm6/311YAxWkdoxAiHZ0ecB7PozI9ifT63vRW1+qS3f246jwENyhdtJDQjL3nkkG6bIBMzGhYoBKmV9h05EQtmMmYpZVIcMyVGs4Y7tuiazhdolG++JPDa/SqetwttZZQE/LakOHSvRIN5B+PuQkALCJmO/Mnk3b58yzx9nlIikLiT0m9vhk2WNSFo5TFkI8HoxRwEf7xnpi0KUpMEkE30Nfmf4M7jZev1uK9Ng5AaIAg9rWlAVhg2CYIf2lRrNx24rnonkoJKjEZ4PNbdIgWmWGFst3jm9lmddA9HJUqT7k/onWB3HweGy3Z+S1btRrtf1ZFx8rcAjq2tIRbHoPKIuJGSZmmJhhYobnwgxjlrPpbLpI1DBRw0QNnyo1TJazs7Wc8WYjD7uDV9HTBpb/2RCvvU+OAGoTo+uCBjl8ZzQ5Mb54Q5fU8O4BF/pOMCgdfPAx7QvQbPYNqqtY3jwPqAuTi8QhPyeHBBvylaPI4oDrzwgjgxNJOLKjcWQaTIowPE4kAk3/kE2HYuX3+Z2UlYKSz6NKE31gbLybgJqw4mwR0kpl/LFWK8ogw7w5oKgIGcWezIe8Pcl9OBoDMGPBx7yptiKF/oTHoQth5VXcAohO6jgHjff5TsMGc0tKbUxITBBULzzo/InIRxRZjfkzCya7HI+vA0rg5GJy8ZmiaDr4kOen61CXojE5nzre5hgQsRdw00XSbedOvpTflPd7B4KXsxp2h9VgI0Uccgzbf7I4HHVNakEHsrpY5wwJbirZ3YowI0252UnIDrk/0CVIzK2YjZYaXQOjLbEs0PJKeQReE8kSlMNXXSsYoeA6I7TaHUDAvWAt4OFYDh3mU0ckFy6CSaGFJ0CFc1MJgJ9pWFNPCLT5skM2VZ9q3lLUzAqEQuqYU8VzlqJTadsPBG3KmlD4TBxwjAi0Jh2Vr3xy8S4si0OBTe2+ZkuhOK9yfWhKWkFtlJAhGm6EPGNaWEIhTXuvBq9C0VYB1drduIAuaLi8lfRj+rejc7V+VR3hTFjWD7E13nnEjQrXheRrTuulSkohg98MWjtgH53Ok/aX7KPJPprso8k+muyjyT7qccjxfDpOymNSHpPymJTHpDyerjx253Fo6XAQp+LJgtYYqiAgg+teiiLF4nnbC48UabcHDtDYh8EWp+ucGaNyOuO/p4ETKzHYSa8jSc10XLPMfeFBDom+b2KR7UgBS+tidHUu6iTl7yfrd4R/BlSzHmw1oAji69KH0cUVTc57p3sWanK2rcvO9Eysef/jEqbwnj1WN43rl1Ust0If3VJ2vC3ahARuefItV3lWKcHZS9hjFcVbSXMtDs+cRLYTJ9G4hkoVe/F+8Kz4ng6P01655AS64ZuwjwVmr+HxiO1SJCE0+gm6B9mfLO3WMU+C0pS3moFiamJWdDkeqwXhFbFJHtqJECrL5SEAmEbKGun68XeewbBLQDX7e+OE5QYcKPjCt4qKSYqvixViphod7+8E6m6wPqgd2RQ7lYFJpjoSW4j30PFzK22Ce0m1hMn3z7IeNo53i/HiWg529z0jclDyQO6hFBRgogzCxNotrw83N8WqyPctkEHdJGChaYS+rBj9qiqWTu7U+ajmIX1uMkksKrGob5VFfSK2hPO6EHg4ccUHMgPWLIMz6WxGKp/ND2pDd+JsNpwm05tlRUmexzZIpiJBFdppzc2sJM3hGUm9UUcFsuyROQqu6HNx2ffM9F3ut9gOeF9st1x6uYBQ4HI2hmbCpCqyTHc6HoP0Sh0OP3JTbDQCFU/eAKuhODqWbG+LnURqccQbqWu1fPwxbcQsZTGeSE9hd/IjhvVT/i0+ESTCW71dZYj9jvROvrVE3nFSyEIDutP06mwYU3LFJVccvYdP3xWXAreOsCB2BG6Npxcp5VGihp+EGj46hQnCuXkXq0dZrF/AXq29cWYhBwVeH8Aj2bzRHmj9c9JxhMv2Pf5hb+YpxBHOe6gsUhGW5/LWRBVjfF8pSeVc9icgKHzQdsxTF

## 9. Select the teacher and cache training summaries

Select a compatible checkpoint by full held-out BERTScore, using ROUGE-L to break ties. The student learns from the teacher’s summaries of training documents.

In [ ]:
use_experiment('model2_adapters')
selected_teacher = select_teacher()


In [ ]:
teacher_cache = cache_teacher_summaries()


## 10. Model 4 — smaller distilled student

Artifacts: `runs/model4_distilled`. Each cell selects this experiment explicitly. Capacity trials and the small-sample fit use disposable model copies; they do not alter saved training checkpoints.


In [ ]:
use_experiment('model4_distilled')
model4_distilled_checks = self_checks()


### Maximum-shape capacity and speed

Trials compare microbatches and activation checkpointing using forward, backward and successful optimizer updates. GPU headroom includes optimizer state. The fastest passing runtime is saved independently for this experiment. Validation/teacher decoding time is additional to the reported training-only estimate.


In [ ]:
use_experiment('model4_distilled')
model4_distilled_capacity = capacity_check()


### Small-sample fit


In [ ]:
use_experiment('model4_distilled')
if RUN_SMALL_SAMPLE_CHECK:
    model4_distilled_fit = small_sample_check()
    if not model4_distilled_fit['loss_decreased']:
        raise RuntimeError('Inspect the small-sample learning check before training.')


### Train or resume

Rerun this training cell after a session pause. `latest.pt` includes optimizer, scheduler, scaler, random state and position. `best.pt` is chosen by generated-summary BERTScore on the same fixed held-out selection subset. It may remain at step 0 if training never beats initialization. Three unimproved epochs stop the run.

An AMP overflow retries the same cases without advancing the data or learning-rate schedule. Repeated failures save the resume state and stop for inspection. For a resume OOM, select this experiment, run `check_resume_memory()`, then use `train_run(micro_batch=1, activation_checkpointing=True)` if it passes.


In [ ]:
use_experiment('model4_distilled')
model4_distilled_training = train_run()
print('Complete:', model4_distilled_training['complete'], flush=True)
if not model4_distilled_training['complete']:
    print('Paused. Rerun this training cell to continue; following cells use the current best.', flush=True)


Parameters: {'total': 70186240, 'trainable': 70186240, 'adapted_modules': 0, 'retained_blocks': [0, 2, 4, 6, 7, 8, 9, 15], 'receptive_field': 2027}
{'event': 'validation', 'step': 0, 'epoch': 0, 'best_bertscore': 0.8049, 'bad_epochs': 0, 'summary_nll': 4.242707924900669, 'perplexity': 69.59605779784557, 'target_tokens': 23910, 'bertscore_f1': 0.8049, 'rouge1_f1': 0.341111, 'rougel_f1': 0.178365}
{'event': 'train', 'step': 1, 'epoch': 0, 'summary_nll': 3.936469554901123, 'lr': 3.992015968063872e-08, 'gradient_norm': 4.7748589515686035, 'skipped': 0, 'elapsed_minutes': 1.4193693262500042}
{'event': 'train', 'step': 20, 'epoch': 0, 'summary_nll': 3.783726692199707, 'lr': 4.191616766467066e-07, 'gradient_norm': 4.707803726196289, 'skipped': 0, 'elapsed_minutes': 1.528909329416668}
{'event': 'train', 'step': 40, 'epoch': 0, 'summary_nll': 3.7412381172180176, 'lr': 8.183632734530938e-07, 'gradient_norm': 4.617387294769287, 'skipped': 0, 'elapsed_minutes': 1.647608777166685}
{'event': 'train'

### Full held-out evaluation against original references


In [ ]:
use_experiment('model4_distilled')
model4_distilled_validation = export_predictions('val')


Professor evaluator: val
Documents: 350
BERTScore F1: 0.826263
ROUGE-1: 0.313350
ROUGE-L: 0.169780

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Analysis and review exports

Exports include per-document scores, source coverage, reference truncation, output length, repetition, training curves and resource usage. Manual review files contain source/reference/generated summaries and blank columns for legal facts, reasoning and the decision. Complete those judgments yourself; BERTScore and lexical overlap do not establish factuality.


In [ ]:
use_experiment('model4_distilled')
model4_distilled_analysis = analyze_validation()


### Test submission and download


In [ ]:
use_experiment('model4_distilled')
model4_distilled_submission = export_predictions('test')


CodaBench submission: /kaggle/working/phase6_model2_student/runs/model4_distilled/test/0dcfd96af948/model4_distilled_submission.zip


Download model4_distilled_submission.zip

## 11. Optional matched gold-only student control

Artifacts: `runs/model4_gold_control`. Each cell selects this experiment explicitly. Capacity trials and the small-sample fit use disposable model copies; they do not alter saved training checkpoints.

This matched gold-only control is optional and defaults to off. Set `RUN_GOLD_CONTROL=True` to run its cells.


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    model4_gold_control_checks = self_checks()


### Maximum-shape capacity and speed

Trials compare microbatches and activation checkpointing using forward, backward and successful optimizer updates. GPU headroom includes optimizer state. The fastest passing runtime is saved independently for this experiment. Validation/teacher decoding time is additional to the reported training-only estimate.


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    model4_gold_control_capacity = capacity_check()


### Small-sample fit


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    if RUN_SMALL_SAMPLE_CHECK:
        model4_gold_control_fit = small_sample_check()
        if not model4_gold_control_fit['loss_decreased']:
            raise RuntimeError('Inspect the small-sample learning check before training.')


### Train or resume

Rerun this training cell after a session pause. `latest.pt` includes optimizer, scheduler, scaler, random state and position. `best.pt` is chosen by generated-summary BERTScore on the same fixed held-out selection subset. It may remain at step 0 if training never beats initialization. Three unimproved epochs stop the run.

An AMP overflow retries the same cases without advancing the data or learning-rate schedule. Repeated failures save the resume state and stop for inspection. For a resume OOM, select this experiment, run `check_resume_memory()`, then use `train_run(micro_batch=1, activation_checkpointing=True)` if it passes.


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    model4_gold_control_training = train_run()
    print('Complete:', model4_gold_control_training['complete'], flush=True)
    if not model4_gold_control_training['complete']:
        print('Paused. Rerun this training cell to continue; following cells use the current best.', flush=True)


### Full held-out evaluation against original references


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    model4_gold_control_validation = export_predictions('val')


### Analysis and review exports

Exports include per-document scores, source coverage, reference truncation, output length, repetition, training curves and resource usage. Manual review files contain source/reference/generated summaries and blank columns for legal facts, reasoning and the decision. Complete those judgments yourself; BERTScore and lexical overlap do not establish factuality.


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    model4_gold_control_analysis = analyze_validation()


### Test submission and download


In [ ]:
if RUN_GOLD_CONTROL:
    use_experiment('model4_gold_control')
    model4_gold_control_submission = export_predictions('test')


## 12. Compare quality, capacity and distillation

The table uses original-reference held-out metrics. The distillation report records teacher/student quality, parameter reduction and checkpoint sizes. Each run's analysis also records GPU memory, training history and generation time. Compare inference timing on the same GPU and batch settings. Decoder and encoder-decoder inputs differ; retain that qualification when discussing architecture effects.

If the optional matched control was not run, a teacher/student gap alone does not measure the causal benefit of KD. Candidate selection and checkpoint selection reuse held-out data, so these are development results; final competition scores are reported separately.


In [ ]:
remaining_comparison = compare_remaining_runs()
